<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/HIDROCARBUROS/PP_Mapa_Lotes_Plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗺️ PERUPETRO · Mapa de lotes con plugins

Es el mismo notebook que `PP_Mapa_Lotes.ipynb` (bloques 1 a 4) más un **Bloque 5** que agrega complementos gráficos al mapa:

| Plugin | Qué hace |
|---|---|
| Fondos CARTO (claro y oscuro) | Fondos neutros en el selector de capas, para que se lean mejor los lotes |
| Nombres de lotes | Rótulo de cada lote desde el zoom 7 (capa «Nombres de lotes») |
| Comunidades discretas | Los grupos de comunidades nativas se ven como círculos morados pequeños |
| Buscador de lotes | Arriba a la izquierda: escribe el lote (p. ej. 95), el mapa va a él y abre su ficha |
| Pantalla completa | Botón arriba a la izquierda |
| Medición | Distancias (km) y áreas (ha) |
| Línea de tiempo animada | Botón «⏱ Evolución histórica»: colorea los lotes por su producción de cada año (bep/d de los productos marcados) con ▶ para animar |

**Uso:** ejecuta los bloques 1 a 5 en orden (el Bloque 2 igual que en el notebook base).
**Resultado:** `DATA/MAPA/PP_mapa_lotes_plus.html`. El Bloque 4 también guarda el mapa base `PP_mapa_lotes.html`, igual al del otro notebook.

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN, DRIVE Y FUNCIONES (ejecutar una vez por sesión) ═══
!pip install geopandas pyogrio folium requests cryptography google-api-python-client --quiet
import os, re, io, json, time, logging, unicodedata
from datetime import date
import requests
import pandas as pd
import geopandas as gpd
import ipywidgets as w
from IPython.display import display
try:                                   # selectores visibles en Colab
    from google.colab import output as _out; _out.enable_custom_widget_manager()
except Exception:
    pass
from shapely.geometry import shape, Point, LineString, MultiLineString, Polygon, MultiPolygon

DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
ruta = "/content/PP_Mapa"; os.makedirs(ruta, exist_ok=True)

# ── Google Drive ─────────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def carpeta(nombre, padre, existentes):
    if nombre not in existentes:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [padre],
                                                        "mimeType": "application/vnd.google-apps.folder"},
                                                  fields="id", supportsAllDrives=True).execute()["id"]
    return existentes[nombre]

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
MAPA_FOLDER_ID = carpeta("MAPA", DRIVE_FOLDER_ID, datos_drive)
mapa_drive = archivos_en(MAPA_FOLDER_ID)
print(f"☁️  Drive conectado · archivos en DATA/MAPA: {len(mapa_drive)}")

# ── Funciones: ArcGIS REST, fuentes externas e indicadores ──────────────────

# ── Conexión ─────────────────────────────────────────────────────────────────
SES = requests.Session()
SES.headers["User-Agent"] = "Mozilla/5.0 (proyecto academico; consulta de datos publicos)"

# Algunos servidores del Estado no envían su certificado intermedio (los navegadores lo completan; Python no).
# Si hay error SSL, se descarga el intermedio desde la dirección AIA del certificado y se reintenta verificando.
_VERIFY = {}
_MUERTOS = set()        # servidores que no responden: no se vuelven a esperar en la misma sesión

def _bundle_con_intermedios(host):
    import ssl, socket, certifi, tempfile
    from cryptography import x509
    from cryptography.hazmat.primitives.serialization import Encoding
    from cryptography.x509.oid import ExtensionOID, AuthorityInformationAccessOID
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    with socket.create_connection((host, 443), timeout=20) as s, ctx.wrap_socket(s, server_hostname=host) as ss:
        cert = x509.load_der_x509_certificate(ss.getpeercert(binary_form=True))
    pems = []
    for _ in range(3):
        try:
            aia = cert.extensions.get_extension_for_oid(ExtensionOID.AUTHORITY_INFORMATION_ACCESS).value
        except x509.ExtensionNotFound:
            break
        urls = [d.access_location.value for d in aia if d.access_method == AuthorityInformationAccessOID.CA_ISSUERS]
        if not urls or cert.issuer == cert.subject:
            break
        datos = requests.get(urls[0], timeout=30).content
        try:
            cert = x509.load_der_x509_certificate(datos)
        except ValueError:
            cert = x509.load_pem_x509_certificate(datos)
        pems.append(cert.public_bytes(Encoding.PEM).decode())
    ruta = os.path.join(tempfile.gettempdir(), f"ca_{host}.pem")
    with open(ruta, "w") as f:
        f.write(open(certifi.where()).read() + "\n" + "\n".join(pems))
    return ruta

def get_json(url, params=None, post=False, intentos=3):
    """GET/POST a un servicio REST de ArcGIS; reintenta y avisa los errores que ArcGIS manda con código 200."""
    from urllib.parse import urlparse
    params = dict(params or {})
    params.setdefault("f", "json")
    host = urlparse(url).hostname
    if host in _MUERTOS:
        raise RuntimeError(f"el servidor {host} no responde (caído o bloqueado)")
    for i in range(1, intentos + 1):
        try:
            kw = {"timeout": (20, 180), "verify": _VERIFY.get(host, True)}
            r = SES.post(url, data=params, **kw) if post else SES.get(url, params=params, **kw)
            r.raise_for_status()
            js = r.json()
            if isinstance(js, dict) and "error" in js:
                cod = js["error"].get("code")
                if cod in (498, 499, 403):
                    raise PermissionError(f"requiere inicio de sesión ({cod})")
                raise RuntimeError(js["error"].get("message", js["error"]))
            return js
        except PermissionError:
            raise
        except requests.exceptions.SSLError as e:
            if host not in _VERIFY:                       # una sola vez por servidor
                try:
                    _VERIFY[host] = _bundle_con_intermedios(host)
                    continue
                except Exception:
                    _VERIFY[host] = True
            if i == intentos:
                raise RuntimeError(f"certificado HTTPS inválido en {host}: {e}") from None
        except (requests.exceptions.ConnectionError, requests.exceptions.ConnectTimeout) as e:
            if i >= 2:                                    # 2 intentos bastan para saber que está caído
                _MUERTOS.add(host)
                raise RuntimeError(f"el servidor {host} no responde (caído o bloqueado): {type(e).__name__}") from None
        except Exception:
            if i == intentos:
                raise
        time.sleep(2 ** i)

def servicios_del_servidor(url_capa):
    """Lista todos los MapServer/FeatureServer del mismo servidor ArcGIS (raíz /rest/services y una carpeta de profundidad)."""
    m = re.match(r"(https?://[^/]+/[^/]+/rest/services)", url_capa, re.I)
    if not m:
        return []
    raiz, out = m.group(1), []
    js = get_json(raiz)
    carpetas = [""] + js.get("folders", [])
    for c in carpetas:
        j = js if not c else get_json(f"{raiz}/{c}")
        for s in j.get("services", []):
            if s.get("type") in ("MapServer", "FeatureServer"):
                out.append(f"{raiz}/{s['name']}/{s['type']}")
    return sorted(set(out))

def norm_txt(t):
    t = unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().upper()

def slug(t):
    return re.sub(r"[^A-Z0-9]+", "_", norm_txt(t)).strip("_")[:60] or "CAPA"

# ── 1. Descubrimiento: aplicación → mapa web → capas ────────────────────────
def _item(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}")

def _item_data(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}/data")

def mapa_web_de_app(portal, app_id):
    """Devuelve (id del mapa web, título) a partir del id de la aplicación (Web AppBuilder, Instant Apps, etc.)."""
    info = _item(portal, app_id)
    if info.get("type") == "Web Map":
        return app_id, info.get("title", "")
    data = _item_data(portal, app_id)
    candidatos = []
    def buscar(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k in ("itemId", "webmap", "webMap", "id") and isinstance(v, str) and re.fullmatch(r"[0-9a-f]{32}", v):
                    candidatos.append(v)
                buscar(v)
        elif isinstance(o, list):
            for v in o:
                buscar(v)
    if isinstance(data.get("map"), dict) and data["map"].get("itemId"):
        candidatos.insert(0, data["map"]["itemId"])
    buscar(data)
    for c in dict.fromkeys(candidatos):
        if c == app_id:
            continue
        try:
            inf = _item(portal, c)
        except Exception:
            continue
        if inf.get("type") == "Web Map":
            return c, inf.get("title", "")
    raise RuntimeError("No encontré el mapa web dentro de la aplicación.")

def capas_de_servicio(url, ruta_titulo=""):
    """Expande un servicio MapServer/FeatureServer o una capa suelta en una lista de capas consultables."""
    url = url.rstrip("/")
    out = []
    if re.search(r"/(MapServer|FeatureServer)$", url, re.I):
        js = get_json(url)
        for l in js.get("layers", []) + js.get("tables", []):
            if l.get("subLayerIds"):
                continue                                  # grupo: sus hijas vienen en la misma lista
            out += capas_de_servicio(f"{url}/{l['id']}", ruta_titulo)
        return out
    js = get_json(url)
    if js.get("type") in ("Group Layer",) or js.get("subLayers") and not js.get("geometryType"):
        for s in js.get("subLayers", []):
            out += capas_de_servicio(f"{url.rsplit('/', 1)[0]}/{s['id']}", ruta_titulo)
        return out
    out.append({"CAPA": js.get("name", ""), "GRUPO": ruta_titulo, "URL": url, "TIPO": js.get("type", ""),
                "GEOMETRIA": js.get("geometryType", ""), "CAMPOS": ", ".join(f["name"] for f in js.get("fields") or []),
                "MAX_REG": js.get("maxRecordCount") or 1000,
                "PAGINA": bool((js.get("advancedQueryCapabilities") or {}).get("supportsPagination"))})
    return out

def capas_del_mapa(portal, webmap_id):
    data = _item_data(portal, webmap_id)
    out = []
    def recorrer(capas, ruta=""):
        for c in capas:
            titulo = c.get("title") or c.get("id") or ""
            r = f"{ruta} / {titulo}" if ruta else titulo
            if c.get("layers") and c.get("layerType") == "GroupLayer":
                recorrer(c["layers"], r); continue
            if c.get("featureCollection"):
                for k, l in enumerate(c["featureCollection"].get("layers", [])):
                    ld = l.get("layerDefinition", {})
                    out.append({"CAPA": ld.get("name") or titulo, "GRUPO": r, "URL": f"featureCollection:{c.get('id')}:{k}",
                                "TIPO": "Colección incrustada", "GEOMETRIA": ld.get("geometryType", ""),
                                "CAMPOS": ", ".join(f["name"] for f in ld.get("fields") or []),
                                "MAX_REG": None, "PAGINA": False, "_FC": l})
                continue
            url = c.get("url")
            if not url and c.get("itemId"):
                try:
                    url = _item(portal, c["itemId"]).get("url")
                except Exception:
                    url = None
            if not url:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": "", "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": "sin URL"}); continue
            try:
                hijas = capas_de_servicio(url, r)
                if c.get("layers") and re.search(r"/MapServer$", url.rstrip("/"), re.I):
                    ids = {str(l.get("id")) for l in c["layers"]}
                    hijas = [h for h in hijas if h["URL"].rsplit("/", 1)[1] in ids] or hijas
                out.extend(hijas)
            except Exception as e:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": url, "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": str(e)[:200]})
    recorrer(data.get("operationalLayers", []))
    return out

def buscar_arcgis(consulta, n=15, portal="https://www.arcgis.com"):
    """Busca servicios públicos en ArcGIS Online (para ANP, comunidades nativas, oleoducto…)."""
    q = f'({consulta}) AND (type:"Feature Service" OR type:"Map Service")'
    js = get_json(f"{portal}/sharing/rest/search", {"q": q, "num": n, "sortField": "numviews", "sortOrder": "desc"})
    return [{"TITULO": r.get("title", ""), "DUEÑO": r.get("owner", ""), "TIPO": r.get("type", ""),
             "URL": r.get("url") or "", "ID": r.get("id"), "VISTAS": r.get("numViews", 0)}
            for r in js.get("results", []) if r.get("url")]

# ── 2. Descarga de una capa ──────────────────────────────────────────────────
def _anillos_a_poligono(rings):
    """Geometría Esri (anillos) → Polygon/MultiPolygon: horario = exterior, antihorario = hueco."""
    ext, huecos = [], []
    for r in rings:
        if len(r) < 4:
            continue
        p = Polygon(r)
        (huecos if p.exterior.is_ccw else ext).append(p)
    if not ext:                                  # anillos mal orientados: todos como exteriores
        ext, huecos = huecos, []
    polys = []
    for e in ext:
        hs = [h.exterior.coords for h in huecos if e.contains(h.representative_point())]
        polys.append(Polygon(e.exterior.coords, hs))
    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

def esri_a_shapely(g):
    if not g:
        return None
    if "x" in g:
        return None if g["x"] is None else Point(g["x"], g["y"])
    if "points" in g:
        from shapely.geometry import MultiPoint
        return MultiPoint(g["points"])
    if "paths" in g:
        return LineString(g["paths"][0]) if len(g["paths"]) == 1 else MultiLineString(g["paths"])
    if "rings" in g:
        return _anillos_a_poligono(g["rings"])
    return None

def gdf_desde_esri(features, wkid):
    filas = [dict(f.get("attributes") or {}) for f in features]
    geoms = [esri_a_shapely(f.get("geometry")) for f in features]
    crs = "EPSG:3857" if wkid in (102100, 102113, 3857, 900913) else f"EPSG:{wkid or 4326}"
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=crs).to_crs(4326)

# ── WFS (GeoServer del geoportal de PERUPETRO) ──────────────────────────────
GML_A_ESRI = {"Polygon": "esriGeometryPolygon", "Line": "esriGeometryPolyline", "Point": "esriGeometryPoint"}

def wfs_capas(base):
    """Capas (FeatureType) publicadas por un WFS: nombre, título y tipo de geometría."""
    import xml.etree.ElementTree as ET
    from urllib.parse import urlparse
    host = urlparse(base).hostname
    r = None
    for i in range(3):
        try:
            r = SES.get(f"{base}/wfs", params={"service": "WFS", "version": "2.0.0", "request": "GetCapabilities"},
                        timeout=(20, 180), verify=_VERIFY.get(host, True))
            break
        except requests.exceptions.SSLError:
            _VERIFY[host] = _bundle_con_intermedios(host)
    r.raise_for_status()
    raiz = ET.fromstring(r.content)
    out = []
    for ft in raiz.iter():
        if ft.tag.endswith("FeatureType"):
            nombre = next((c.text for c in ft if c.tag.endswith("Name")), None)
            titulo = next((c.text for c in ft if c.tag.endswith("Title")), None) or nombre
            if nombre:
                out.append({"CAPA": titulo, "GRUPO": "Geoportal PERUPETRO (WFS)", "URL": f"{base}/wfs#{nombre}",
                            "TIPO": "WFS", "GEOMETRIA": "", "CAMPOS": "", "MAX_REG": 5000, "PAGINA": True, "WFS": nombre})
    return out

def wfs_describir(base, capas):
    """Completa geometría y campos de varias capas WFS con UNA sola petición DescribeFeatureType."""
    capas = capas if isinstance(capas, list) else [capas]
    js = get_json(f"{base}/wfs", {"service": "WFS", "version": "2.0.0", "request": "DescribeFeatureType",
                                  "typeNames": ",".join(c["WFS"] for c in capas), "outputFormat": "application/json"})
    por_nombre = {}
    for ft in js.get("featureTypes") or []:
        por_nombre[ft.get("typeName")] = ft
        por_nombre[f"{js.get('targetPrefix', '')}:{ft.get('typeName')}"] = ft
    for c in capas:
        ft = por_nombre.get(c["WFS"]) or por_nombre.get(c["WFS"].split(":")[-1]) or (
            (js.get("featureTypes") or [{}])[0] if len(capas) == 1 else {})
        props = ft.get("properties", [])
        geo = next((p.get("localType") or p.get("type", "") for p in props if str(p.get("type", "")).startswith("gml:")), "")
        c["GEOMETRIA"] = next((v for k, v in GML_A_ESRI.items() if k in geo), "")
        c["CAMPOS"] = ", ".join(p["name"] for p in props if not str(p.get("type", "")).startswith("gml:"))
    return capas

def wfs_descargar(base, nombre, paso=5000):
    """Todas las entidades de una capa WFS en GeoJSON (EPSG:4326), paginando; quita la coordenada Z."""
    import shapely
    partes, inicio = [], 0
    while True:
        js = get_json(f"{base}/wfs", {"service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": nombre,
                                      "outputFormat": "application/json", "srsName": "EPSG:4326",
                                      "count": paso, "startIndex": inicio})
        feats = js.get("features", [])
        if feats:
            g = gpd.GeoDataFrame.from_features(feats, crs=4326)
            g["geometry"] = g.geometry.map(lambda x: shapely.force_2d(x) if x is not None else x)
            partes.append(g)
        inicio += len(feats)
        total = js.get("numberMatched") or js.get("totalFeatures")
        if not feats or len(feats) < paso or (isinstance(total, int) and inicio >= total):
            break
    return gpd.GeoDataFrame(pd.concat(partes, ignore_index=True), crs=4326) if partes else gpd.GeoDataFrame(geometry=[], crs=4326)

def descargar_capa(capa, aviso=print):
    """Descarga todas las entidades de una capa en EPSG:4326 (WFS, GeoJSON de ArcGIS o JSON de Esri)."""
    if capa.get("WFS"):
        return wfs_descargar(capa["URL"].split("/wfs#")[0], capa["WFS"])
    if capa.get("_FC"):
        fs = capa["_FC"].get("featureSet", {})
        wkid = ((capa["_FC"].get("layerDefinition", {}).get("extent") or {}).get("spatialReference") or {}).get("wkid") \
               or (fs.get("spatialReference") or {}).get("wkid") or 102100
        return gdf_desde_esri(fs.get("features", []), wkid)
    url = capa["URL"].rstrip("/") + "/query"
    base = {"where": "1=1", "outFields": "*", "outSR": 4326, "returnGeometry": "true"}
    total = get_json(url, {**base, "returnCountOnly": "true"}, post=True).get("count")
    paso = int(capa.get("MAX_REG") or 1000)
    wk = lambda js: (js.get("spatialReference") or {}).get("wkid", 4326)
    partes = []
    def por_ids():
        ids = sorted(get_json(url, {"where": "1=1", "returnIdsOnly": "true"}, post=True).get("objectIds") or [])
        for i in range(0, len(ids), paso):
            js = get_json(url, {**base, "where": "", "objectIds": ",".join(map(str, ids[i:i + paso]))}, post=True)
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    if capa.get("PAGINA"):
        off = 0
        while True:
            js = get_json(url, {**base, "resultOffset": off, "resultRecordCount": paso}, post=True)
            feats = js.get("features", [])
            if feats:
                partes.append(gdf_desde_esri(feats, wk(js)))
            off += len(feats)
            if not feats or (total is not None and off >= total) or (total is None and not js.get("exceededTransferLimit")):
                break
    elif total and total > paso:
        por_ids()
    else:
        js = get_json(url, base, post=True)
        if js.get("exceededTransferLimit"):
            por_ids()
        else:
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    gdf = pd.concat(partes, ignore_index=True) if partes else gpd.GeoDataFrame(geometry=[], crs=4326)
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=4326)
    if total is not None and len(gdf) != total:
        aviso(f"   ⚠️ {capa['CAPA']}: el servidor declara {total} entidades y llegaron {len(gdf)}")
    return gdf

# ── 3. Fuentes externas ──────────────────────────────────────────────────────
GEOBOUNDARIES_API = "https://www.geoboundaries.org/api/current/gbOpen/PER/ADM1/"
NE_RIOS = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_rivers_lake_centerlines.zip"
HYDRORIVERS_SA = "https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_sa_shp.zip"
OVERPASS = "https://overpass-api.de/api/interpreter"
BBOX_PERU = (-81.5, -18.5, -68.5, 0.1)                     # lon_min, lat_min, lon_max, lat_max

def bajar(url, destino, aviso=print):
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return destino
    with SES.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(destino + ".part", "wb") as fh:
            for b in r.iter_content(1 << 20):
                fh.write(b)
    os.replace(destino + ".part", destino)
    return destino

def departamentos(carpeta):
    js = SES.get(GEOBOUNDARIES_API, timeout=60).json()
    ruta = bajar(js["gjDownloadURL"], os.path.join(carpeta, "geoBoundaries-PER-ADM1.geojson"))
    g = gpd.read_file(ruta).to_crs(4326)
    g["DEPARTAMENTO"] = g["shapeName"].map(norm_txt).str.replace(r"^(REGION|DEPARTAMENTO) (DE )?", "", regex=True)
    g["DEPARTAMENTO"] = g["DEPARTAMENTO"].replace({"LIMA PROVINCE": "LIMA", "PROVINCIA DE LIMA": "LIMA",
                                                   "LIMA METROPOLITANA": "LIMA", "CALLAO PROVINCE": "CALLAO",
                                                   "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO"})
    return g.dissolve("DEPARTAMENTO", as_index=False)[["DEPARTAMENTO", "geometry"]]

def rios_hydrorivers(carpeta, caudal_min):
    """Ríos de HydroRIVERS (Sudamérica) con caudal medio ≥ caudal_min m³/s, recortados a Perú."""
    import zipfile
    ruta = os.path.join(carpeta, "HydroRIVERS_v10_sa_shp.zip")
    if os.path.exists(ruta) and not zipfile.is_zipfile(ruta):
        os.remove(ruta)                                   # descarga anterior inválida (p. ej. una página HTML)
    bajar(HYDRORIVERS_SA, ruta)
    if not zipfile.is_zipfile(ruta):
        ini = open(ruta, "rb").read(200)
        os.remove(ruta)
        raise RuntimeError(f"HydroRIVERS no devolvió un ZIP (empieza con {ini[:60]!r}); vuelve a intentar más tarde")
    shp = [n for n in zipfile.ZipFile(ruta).namelist() if n.lower().endswith(".shp")]
    if not shp:
        raise RuntimeError("el ZIP de HydroRIVERS no contiene un .shp")
    g = gpd.read_file(f"zip://{ruta}!{shp[0]}", bbox=BBOX_PERU)   # el .shp está dentro de una carpeta del ZIP
    return g[g["DIS_AV_CMS"] >= caudal_min][["HYRIV_ID", "DIS_AV_CMS", "ORD_FLOW", "geometry"]].to_crs(4326)

def rios_osm_nombres():
    """Ríos principales de OpenStreetMap con nombre (relaciones waterway=river): consulta liviana."""
    q = """[out:json][timeout:300];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    relation["waterway"="river"]["name"](area.pe);out geom;"""
    js = overpass(q)
    filas, geoms = [], []
    for e in js.get("elements", []):
        lineas = [LineString([(p["lon"], p["lat"]) for p in m_["geometry"]]) for m_ in e.get("members", [])
                  if m_.get("type") == "way" and len(m_.get("geometry") or []) >= 2]
        if lineas:
            filas.append({"RIO": e.get("tags", {}).get("name", "")}); geoms.append(MultiLineString(lineas))
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=4326)

def rios_natural_earth(carpeta):
    ruta = bajar(NE_RIOS, os.path.join(carpeta, "ne_10m_rivers_lake_centerlines.zip"))
    g = gpd.read_file(f"zip://{ruta}", bbox=BBOX_PERU).to_crs(4326)
    g["RIO"] = g["name"].fillna(g.get("name_es")).fillna("")
    return g[["RIO", "geometry"]]

OVERPASS_SERVIDORES = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
                       "https://overpass.private.coffee/api/interpreter"]

def overpass(consulta):
    """Consulta Overpass (OpenStreetMap) probando varios servidores; se identifica como pide su política de uso."""
    cab = {"User-Agent": "scraper-mef/1.0 (proyecto academico; github.com/adriansivina-droid/scraper-mef)",
           "Accept": "application/json,*/*"}
    errores = []
    for srv in OVERPASS_SERVIDORES:
        try:
            r = requests.post(srv, data={"data": consulta}, headers=cab, timeout=900)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            errores.append(f"{srv.split('/')[2]}: {str(e)[:80]}")
    raise RuntimeError("Overpass no respondió · " + " | ".join(errores))

def _osm_a_gdf(js, tipo):
    filas, geoms = [], []
    for e in js.get("elements", []):
        t = e.get("tags", {})
        if e["type"] == "way" and e.get("geometry"):
            coords = [(p["lon"], p["lat"]) for p in e["geometry"]]
            if tipo == "linea":
                g = LineString(coords)
            elif len(coords) >= 4 and coords[0] == coords[-1]:
                g = Polygon(coords)
            else:
                continue
        elif e["type"] == "relation" and e.get("members") and tipo == "poligono":
            ext = [[(p["lon"], p["lat"]) for p in m.get("geometry", [])] for m in e["members"]
                   if m.get("role") in ("outer", "") and m.get("geometry")]
            from shapely.ops import linemerge, polygonize, unary_union
            polys = list(polygonize(unary_union([LineString(c) for c in ext if len(c) >= 2])))
            if not polys:
                continue
            g = unary_union(polys)
        else:
            continue
        filas.append({"NOMBRE": t.get("name", ""), "OSM_ID": f"{e['type']}/{e['id']}", **{k: t.get(k, "") for k in
                     ("protect_class", "protection_title", "operator", "substance")}})
        geoms.append(g)
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=4326)

def anp_osm():
    q = """[out:json][timeout:500];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["boundary"="protected_area"](area.pe);relation["boundary"="protected_area"](area.pe);
     way["leisure"="nature_reserve"](area.pe);relation["leisure"="nature_reserve"](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "poligono")

def oleoducto_osm():
    q = """[out:json][timeout:300];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["man_made"="pipeline"]["substance"~"oil|crude",i](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "linea")

# ── 4. Indicadores por lote ──────────────────────────────────────────────────
CRS_AREA = "ESRI:102033"        # Albers equivalente de Sudamérica (áreas en km²)
CRS_DIST = "EPSG:32718"         # UTM 18S (distancias en km; error < 1 % en la Amazonía peruana)
IQUITOS = Point(-73.2538, -3.7491)

_ROM = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

def _romano(s):
    if not s or not re.fullmatch(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})", s):
        return None
    total = 0
    for i, ch in enumerate(s):
        v = _ROM[ch]
        total += -v if i + 1 < len(s) and _ROM[s[i + 1]] > v else v
    return total

def lote_clave(t):
    """'LOTE 192' → '192'; 'LOTE_Z62' → 'Z-62'; 'Lote Z - 2B' → 'Z-2B'; '1 AB' → '1-AB'.
    Los lotes de la selva que vienen en números romanos (XCV, CIII, LXVII, VIII) pasan a arábigos; los del noroeste
    (I a XXIX) se mantienen en romanos, como en la producción."""
    s = norm_txt(str(t).replace("_", " "))
    s = re.sub(r"^(LOTE|BLOQUE|AREA|CONTRATO)\s+", "", s)
    s = re.sub(r"\s*-\s*", "-", s).replace(" ", "")
    s = re.sub(r"^([A-Z]{1,2})(\d)", r"\1-\2", s) if re.match(r"^(Z|XX?|CO)\d", s) else s
    s = re.sub(r"^(\d+)([A-Z]+)$", r"\1-\2", s) if re.match(r"^\d+[A-Z]{1,2}$", s) else s
    n = _romano(s)
    if n is not None and (n >= 30 or n == 8):          # 8 (VIII) y ≥ 30 son lotes de la selva numerados en arábigo
        s = str(n)
    return s

def campo_lote(gdf):
    cands = [c for c in gdf.columns if c != "geometry"]
    for pat in (r"^LOTE$", r"LOTE", r"^(NOMBRE|NAME|NOM)$", r"NOMBRE|NAME|CONTRATO|BLOQUE"):
        for c in cands:
            if re.search(pat, norm_txt(c)) and gdf[c].astype(str).str.len().gt(0).mean() > 0.5:
                return c
    return None

def campo(gdf, *patrones):
    for p in patrones:
        for c in gdf.columns:
            if c != "geometry" and re.search(p, norm_txt(c)):
                return c
    return None

def reparto_por_poligonos(lotes, capa, nombre_col, prefijo):
    """% del área de cada lote dentro de cada polígono de `capa` → principal, detalle y suma."""
    if capa is None or capa.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_AREA)
    b = capa[[nombre_col, "geometry"]].to_crs(CRS_AREA)
    b = b[b.geometry.notna() & ~b.geometry.is_empty].copy()
    b["geometry"] = b.geometry.buffer(0)
    a = a.copy(); a["geometry"] = a.geometry.buffer(0); a["_ID"] = a.index
    inter = gpd.overlay(a, b, how="intersection", keep_geom_type=True)
    inter["_KM2"] = inter.geometry.area / 1e6
    area = a.geometry.area / 1e6
    res = {}
    for i, g in inter.groupby("_ID"):
        s = g.groupby(nombre_col)["_KM2"].sum().sort_values(ascending=False)
        pct = (s / area.loc[i] * 100).round(1)
        res[i] = {f"{prefijo}": s.index[0], f"{prefijo}_PCT": min(pct.iloc[0], 100.0),
                  f"{prefijo}_DETALLE": "; ".join(f"{k} {v:.1f}%" for k, v in pct.items() if v >= 0.1),
                  f"{prefijo}_N": int((pct >= 0.1).sum()), f"{prefijo}_PCT_TOTAL": min(round(pct.sum(), 1), 100.0)}
    return pd.DataFrame.from_dict(res, orient="index").reindex(lotes.index)

def distancia_km(lotes, capa, nombre_col=None):
    """Distancia mínima (km) de cada lote a la capa (0 si la toca) y, si hay nombre, el elemento más cercano."""
    if capa is None or capa.empty:
        return pd.Series(index=lotes.index, dtype=float), pd.Series(index=lotes.index, dtype=object)
    a = lotes.to_crs(CRS_DIST)
    b = capa[capa.geometry.notna()].to_crs(CRS_DIST)
    j = gpd.sjoin_nearest(a[["geometry"]], b[[c for c in [nombre_col] if c] + ["geometry"]], how="left",
                          distance_col="_D")
    j = j[~j.index.duplicated()]
    return (j["_D"] / 1000).round(1), (j[nombre_col] if nombre_col else pd.Series(index=lotes.index, dtype=object))

def comunidades(lotes, ccnn, nombre_col, radio_km=5):
    """Comunidades nativas dentro del lote y a menos de `radio_km` km (cuenta y nombres)."""
    if ccnn is None or ccnn.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_DIST)
    b = ccnn[[nombre_col, "geometry"]].to_crs(CRS_DIST)
    b = b[b.geometry.notna()].copy()
    pol = b.geom_type.isin(["Polygon", "MultiPolygon"])
    b.loc[pol, "geometry"] = b.loc[pol, "geometry"].buffer(0)
    out = {}
    dentro = gpd.sjoin(a, b, predicate="intersects")
    cerca = gpd.sjoin(gpd.GeoDataFrame(geometry=a.geometry.buffer(radio_km * 1000), crs=a.crs), b, predicate="intersects")
    for i in lotes.index:
        d = dentro[dentro.index == i][nombre_col].dropna().astype(str).unique()
        c = cerca[cerca.index == i][nombre_col].dropna().astype(str).unique()
        out[i] = {"CCNN_DENTRO_N": len(d), f"CCNN_{radio_km}KM_N": len(c),
                  "CCNN_DENTRO": "; ".join(sorted(d)[:40]) + (" …" if len(d) > 40 else "")}
    return pd.DataFrame.from_dict(out, orient="index")


# ── 5. Producción por lote y por campo ───────────────────────────────────────
import calendar

def campos(geo):
    """Agrupa los lotes que comparten polígono (1-AB/192, 31-B/31-D/31-B/D…) en un mismo campo: componentes conexas."""
    padre = {}
    def raiz(x):
        while padre.setdefault(x, x) != x:
            padre[x] = padre[padre[x]]; x = padre[x]
        return x
    for lote, gd in zip(geo["LOTE"], geo["GEOMETRIA_DE"].astype(str)):
        for p in gd.split("+"):
            padre[raiz(lote)] = raiz(p)
    grupos = {}
    for lote in geo["LOTE"]:
        grupos.setdefault(raiz(lote), set()).add(lote)
    return {lote: grupo for grupo in grupos.values() for lote in grupo}

def _dias(t):                      # t = año*12 + mes-1
    return calendar.monthrange(t // 12, t % 12 + 1)[1]

def _mm_aaaa(t):
    return f"{t % 12 + 1:02d}/{t // 12}"

def indicadores_produccion(prod, clave, grupo=None):
    """Indicadores por lote o campo a partir de la producción mensual (todos los líquidos fiscalizados).
    prod: AÑO, MES, PRODUCTO, OPERADOR, BARRILES y la columna `clave`.
    grupo: columna del campo; los meses en que el campo produjo con otro contrato no cuentan como paralización."""
    p = prod.copy()
    p["T"] = p["AÑO"].astype(int) * 12 + p["MES"].astype(int) - 1
    fin, ini_datos = int(p["T"].max()), int(p["T"].min())
    anio_fin_completo = fin // 12 if fin % 12 == 11 else fin // 12 - 1
    activo_grupo = ({k: set(g.loc[g["BARRILES"] > 0, "T"]) for k, g in p.groupby(grupo)} if grupo else {})
    out = {}
    for k, g in p.groupby(clave):
        m = g.groupby("T")["BARRILES"].sum()
        pos = m[m > 0]
        prodp = g.groupby("PRODUCTO")["BARRILES"].sum()
        base = {"PETROLEO_ACUM_MMBBL": round(float(prodp.get("PETROLEO", 0)) / 1e6, 3),
                "LIQUIDOS_ACUM_MMBBL": round(float(prodp.sum()) / 1e6, 3),
                "PRODUCTO_PRINCIPAL": prodp.idxmax() if prodp.sum() > 0 else ""}
        if pos.empty:
            out[k] = {**base, "ESTADO_PRODUCCION": "SIN PRODUCCIÓN REGISTRADA"}; continue
        t0, t1 = int(pos.index.min()), int(pos.index.max())
        serie = m.reindex(range(t0, fin + 1), fill_value=0.0)
        bpd_m = pd.Series({t: v / _dias(t) for t, v in serie.items()})
        anual, completos = {}, []
        for a in range(t0 // 12, fin // 12 + 1):
            ts = [t for t in serie.index if t // 12 == a]
            anual[a] = sum(serie[t] for t in ts) / sum(_dias(t) for t in ts)
            if len(ts) == 12:
                completos.append(a)
        anual = pd.Series(anual)
        completos_pos = [a for a in completos if anual[a] > 0]
        pico = anual[completos_pos] if completos_pos else anual   # el pico se busca en años completos con producción
        ceros_anio = {a: sum(1 for t in serie.index if t // 12 == a and serie[t] == 0) for a in anual.index}
        ult12 = [t for t in serie.index if t > fin - 12]
        bpd_act = sum(serie[t] for t in ult12) / sum(_dias(t) for t in ult12)
        # declinación: variación anual compuesta de la producción diaria en 5 años, entre dos años "normales"
        # (≤ 2 meses sin producción) para que una paralización (2016, 2020…) no la distorsione
        y_fin = min(anio_fin_completo, t1 // 12 if t1 % 12 == 11 else t1 // 12 - 1)
        decl, periodo = None, ""
        for y1 in range(y_fin, y_fin - 4, -1):
            y0 = y1 - 5
            if all(y in anual and y in completos and anual[y] > 0 and ceros_anio[y] <= 2 for y in (y0, y1)):
                decl, periodo = round(((anual[y1] / anual[y0]) ** (1 / 5) - 1) * 100, 1), f"{y0}–{y1}"
                break
        cubiertos = activo_grupo.get(g[grupo].iloc[0], set()) if grupo else set()
        ceros = ((serie == 0) & ~serie.index.isin(list(cubiertos))).astype(int)
        racha, mejor, inicio, mejor_ini = 0, 0, None, None
        for t, c in ceros.items():
            if c:
                racha += 1; inicio = t if racha == 1 else inicio
                if racha > mejor:
                    mejor, mejor_ini = racha, inicio
            else:
                racha = 0
        ops = (g[(g["BARRILES"] > 0) & (g["OPERADOR"].fillna("").astype(str).str.strip() != "")].groupby("OPERADOR")["AÑO"].agg(["min", "max"]).sort_values(["min", "max"]))
        out[k] = {**base,
                  "ESTADO_PRODUCCION": "EN PRODUCCIÓN" if t1 > fin - 12 else "PARALIZADO",
                  "PRIMERA_PRODUCCION": ("≤" if t0 == ini_datos else "") + _mm_aaaa(t0),
                  "ULTIMA_PRODUCCION": _mm_aaaa(t1),
                  "BPD_ACTUAL": int(round(float(bpd_act))),
                  "PICO_AÑO": int(pico.idxmax()), "PICO_BPD": int(round(float(pico.max()))),
                  "DECLINACION_ANUAL_5A_PCT": None if decl is None else float(decl),
                  "DECLINACION_PERIODO": periodo,
                  "MESES_SIN_PRODUCCION": int(ceros.sum()),
                  "PARALIZACION_MAX_MESES": int(mejor),
                  "PARALIZACION_MAX_DESDE": _mm_aaaa(mejor_ini) if mejor_ini is not None else "",
                  "OPERADORES": " → ".join(f"{o} ({int(r['min'])}" + (f"–{int(r['max'])})" if r['max'] != r['min'] else ")")
                                          for o, r in ops.iterrows()),
                  "_SERIE_ANUAL_MMBBL": {int(a): round(float(v), 3) for a, v in (g.groupby("AÑO")["BARRILES"].sum() / 1e6).items()}}
    return out

def sparkline_svg(serie, anio_ini=None, anio_fin=None, ancho=220, alto=46, color="#b30000"):
    """Mini gráfico SVG de barras con la producción anual (millones de barriles), en el mismo eje de años para todos."""
    if not serie:
        return ""
    anios = list(range(anio_ini or min(serie), (anio_fin or max(serie)) + 1)); vmax = max(serie.values()) or 1
    w = ancho / len(anios)
    barras = "".join(f'<rect x="{i * w:.1f}" y="{alto - 12 - (serie.get(a, 0) / vmax) * (alto - 16):.1f}" width="{max(w - 1, 1):.1f}" '
                     f'height="{(serie.get(a, 0) / vmax) * (alto - 16):.1f}" fill="{color}"><title>{a}: {serie.get(a, 0):.2f} MM bbl</title></rect>'
                     for i, a in enumerate(anios))
    return (f'<svg width="{ancho}" height="{alto}" xmlns="http://www.w3.org/2000/svg">{barras}'
            f'<text x="0" y="{alto - 1}" font-size="9" fill="#555">{anios[0]}</text>'
            f'<text x="{ancho}" y="{alto - 1}" font-size="9" fill="#555" text-anchor="end">{anios[-1]}</text></svg>')


# ── 6. Operadores (misma homologación que el notebook de producción) ─────────
# Nombres de operadores: el PDF corta los nombres largos en dos renglones y tiene erratas; se homologan.
# Se conserva el nombre leído en OPERADOR_ORIGINAL.
OPERADOR_NORM = [
    (r"^PLUSPETRO$|^PLUSPETROL PERU( CORP\.?)?$", "PLUSPETROL"),
    (r"^PPN( EN( LIQUIDACI[OÓ]N)?)?$", "PLUSPETROL NORTE (EN LIQUIDACIÓN)"),
    (r"^UPLAND( OIL( AND( GAS)?)?)?$", "UPLAND OIL AND GAS"),
    (r"^CORPORACI[OÓ]N DEL( SUR)?$", "CORPORACIÓN DEL SUR"),
    (r"^PETROLERA MONTERR+ICO$|^PETROMONT$", "PETROLERA MONTERRICO"),   # PETROMONT = su nombre corto
    (r"^UNIENE?R?G[IÍ]A( ABC)?$", "UNIENERGÍA ABC"),
    (r"^UNIPETRO( ABC)?$", "UNIPETRO ABC"),
    (r"^PETRO-?TECH$", "PETROTECH"),
    (r"^G\.? ?M\.? ?P\.?( S\.? ?A\.?)?$", "GMP"),
    (r"^PETROPER[UÚ]$", "PETROPERÚ"),
    (r"^VIGO( ENERGY)?$", "VIGO ENERGY"),
    (r"^PACIFIC( STRATUS)?$", "PACIFIC STRATUS"),
    (r"^(DPB|BPD|M?M?BBL|LBB)$", ""),               # rótulos de gráficos leídos como operador
]

def fecha_dmy(v):
    """Fecha de un atributo (epoch en milisegundos de ArcGIS, ISO del WFS o texto) → dd/mm/aaaa."""
    if v is None or (isinstance(v, float) and v != v) or str(v).strip() in ("", "None", "nan", "NaT"):
        return ""
    x = str(v).strip()
    try:
        if re.fullmatch(r"-?\d{9,14}(\.0+)?", x):                     # epoch ms (o s)
            n = float(x)
            t = pd.to_datetime(n, unit="ms" if abs(n) > 1e11 else "s", utc=True)
        elif re.match(r"\d{4}-\d{2}-\d{2}", x):                       # ISO
            t = pd.to_datetime(x[:10])
        elif re.fullmatch(r"\d{1,2}[/.-]\d{1,2}[/.-]\d{4}", x):       # ya viene día/mes/año
            t = pd.to_datetime(x, dayfirst=True)
        else:
            return x
        return t.strftime("%d/%m/%Y")
    except Exception:
        return x

def norm_operador(nombre):
    """Homologa un nombre de operador; los consorcios 'A / B' se homologan por partes."""
    partes = []
    if nombre is None or (isinstance(nombre, float) and nombre != nombre):    # None / NaN
        return ""
    for parte in str(nombre).split("/"):
        x = re.sub(r"\s+", " ", parte).strip().upper()
        for patron, bueno in OPERADOR_NORM:
            if re.match(patron, x):
                x = bueno
                break
        if x and x not in ("NAN", "NONE"):
            partes.append(x)
    return " / ".join(partes)



print("✅ Funciones listas")

In [ ]:
# ═══ BLOQUE 2 · DESCUBRIR CAPAS Y ELEGIR FUENTES ═══
PORTAL = "https://perupetro.maps.arcgis.com"
APP_ID = "cc883917c55a4264a01debfa02585e2e"          # visor "mapa de lotes" de PERUPETRO
URLS_EXTRA = []        # opcional: URLs de servicios MapServer/FeatureServer adicionales (una por elemento)
BUSQUEDAS = {          # búsquedas en ArcGIS Online (respaldo y capas que no son de PERUPETRO)
    "LOTES": "lotes hidrocarburos Peru",
    "LOTES2": "lotes de contrato PERUPETRO",
    "LOTES3": "lotes petroleros Peru",
    "LOTES4": "contratos hidrocarburos Peru lotes",
    "ANP": "areas naturales protegidas SERNANP",
    "CCNN": "comunidades nativas Peru",
    "OLEODUCTO": "oleoducto norperuano",
}
GEOM_TEMA = {"LOTES": ("Polygon",), "LOTES2": ("Polygon",), "LOTES3": ("Polygon",), "LOTES4": ("Polygon",), "ANP": ("Polygon",), "CCNN": ("Polygon", "Point"), "OLEODUCTO": ("Polyline",)}

print("🔎 Leyendo el visor de PERUPETRO…")
WEBMAP_ID, WEBMAP_TIT = mapa_web_de_app(PORTAL, APP_ID)
print(f"   Mapa web: «{WEBMAP_TIT}» ({WEBMAP_ID})")
capas_pp = capas_del_mapa(PORTAL, WEBMAP_ID)
for u in URLS_EXTRA:
    try:
        capas_pp += capas_de_servicio(u, "EXTRA")
    except Exception as e:
        print(f"   ⚠️ {u}: {e}")
for c in capas_pp:
    c["FUENTE"], c["TEMA"] = "PERUPETRO", ""

# Geoportal oficial de PERUPETRO (GeoServer): WFS con lotes de contrato vigentes, cuencas, ANP, comunidades…
GEOSERVER = "https://geoportal.perupetro.com.pe/geoserver"
UTILES_WFS = r"LOTES?_CONTRATO$|CUENCAS?.SEDIMENT|AREAS?.NATURAL|COMUNIDAD|PUEBLOS?.INDIGENA|RESERVAS?.TERRITORIAL|OLEODUCTO|^DUCTOS$|YACIMIENTOS$|AREA.PROMOCION|CONVENIO|ARCOS?$"
EXCLUIR_WFS = r"_(INICIO|FIN|F|NW|PASH|PUNTOS)$|ACUEDUCTO|GASODUCTO|POLIDUCTO"
TEMAS_GEOPORTAL = set()
try:
    wfs = wfs_capas(GEOSERVER)
    utiles = [c for c in wfs if re.search(UTILES_WFS, norm_txt(c["WFS"].split(":")[-1]).replace(" ", "_"))
              and not re.search(EXCLUIR_WFS, norm_txt(c["WFS"].split(":")[-1]).replace(" ", "_"))]
    try:
        wfs_describir(GEOSERVER, utiles)                 # una sola petición para todas
    except Exception:
        for c in utiles:                                  # respaldo: una por una
            try:
                wfs_describir(GEOSERVER, c)
            except Exception as e:
                c["ERROR"] = f"DescribeFeatureType: {e}"[:200]
    for c in utiles:
        c["FUENTE"], c["TEMA"] = "PERUPETRO", ""
        t = norm_txt(c["WFS"])
        TEMAS_GEOPORTAL |= {k for k, p in [("ANP", r"NATURAL"), ("CCNN", r"COMUNIDAD|INDIGENA"), ("OLEODUCTO", r"OLEODUCTO")]
                            if re.search(p, t)}
    capas_pp = [c for c in utiles if c["GEOMETRIA"]] + capas_pp      # el geoportal va primero (prioridad)
    print(f"🛰️ Geoportal PERUPETRO (WFS): {len(wfs)} capas publicadas · {len(utiles)} útiles: "
          f"{', '.join(c['WFS'].split(':')[-1] for c in utiles)}")
except Exception as e:
    print(f"⚠️ Geoportal PERUPETRO (WFS) no disponible: {e}")

# Si la capa del visor no responde, se revisan los demás servicios del mismo servidor de PERUPETRO
fallidas = [c for c in capas_pp if c.get("ERROR")]
if fallidas:
    print("\n⚠️ Capas del visor que no responden (también fallan en el navegador):")
    for c in fallidas:
        print(f"   · {c['CAPA']}: {c['ERROR']}\n     {c['URL']}")
    vistos = set()
    for c in fallidas:
        try:
            servs = servicios_del_servidor(c["URL"])
        except Exception as e:
            print(f"   ↳ No se pudo listar el servidor: {e}"); continue
        print(f"   ↳ El servidor responde y tiene {len(servs)} servicios; reviso los de lotes/hidrocarburos…")
        for u in servs:
            if u in vistos or not re.search(r"LOTE|CONTRAT|MAPA.?BASE|HIDROCARB|BLOQUE", norm_txt(u)):
                continue
            vistos.add(u)
            try:
                for h in capas_de_servicio(u, u.split("/services/")[1]):
                    h["FUENTE"], h["TEMA"] = "PERUPETRO", ""; capas_pp.append(h)
            except Exception as e:
                print(f"     · {u}: {e}")

PATRON_TEMA = {"LOTES": r"\bLOTES?\b|CONTRAT|BLOQUES? PETROL|HIDROCARB",
               "ANP": r"AREAS? NATURAL|PROTEGID|\bANP\b|SERNANP|RESERVA|PARQUE NACIONAL",
               "CCNN": r"COMUNIDAD(ES)? NATIVA|CC\.?NN|PUEBLOS? INDIGENA|INDIGEN",
               "OLEODUCTO": r"OLEODUCTO|\bONP\b|NORPERUANO|DUCTO"}
def texto(c):
    return norm_txt(f"{c['CAPA']} {c['GRUPO']} {c['URL']} {c['FUENTE']}")

capas_ext = []
for tema, q in BUSQUEDAS.items():
    if tema.rstrip("234") in TEMAS_GEOPORTAL:
        continue                                          # el geoportal oficial ya trae este tema
    try:
        res = buscar_arcgis(q)
    except Exception as e:
        print(f"   ⚠️ búsqueda {tema}: {e}"); res = []
    base = tema.rstrip("234")
    for r in res[:5]:
        try:
            for c in capas_de_servicio(r["URL"], r["TITULO"]):
                c.update({"FUENTE": f"ArcGIS Online · {r['DUEÑO']}", "TEMA": base})
                # solo capas cuyo nombre corresponde al tema (evita cientos de capas ajenas del mismo servicio)
                if (any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema]) and re.search(PATRON_TEMA[base], texto(c))
                        and c["URL"] not in {x["URL"] for x in capas_ext}):
                    capas_ext.append(c)
        except Exception as e:
            pass


def etiqueta(c):
    g = c["GEOMETRIA"].replace("esriGeometry", "")
    return f"{c['CAPA']}  [{g}]  ·  {c['GRUPO'][:50]}  ·  {c['FUENTE'][:40]}"

capas = capas_pp + capas_ext
for i, c in enumerate(capas):
    c["N"] = i
POR_N_ = {c["N"]: c for c in capas}
cat = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in capas])
pd.set_option("display.max_colwidth", 70)
print(f"\n📋 Capas de PERUPETRO: {len(capas_pp)}")
display(cat[cat["FUENTE"] == "PERUPETRO"][["N", "CAPA", "GRUPO", "GEOMETRIA", "TIPO"] + (["ERROR"] if "ERROR" in cat else [])])
print(f"\n📋 Candidatas externas (filtradas por tema): {len(capas_ext)}")
if capas_ext:
    with pd.option_context("display.max_rows", 200):
        display(cat[cat["FUENTE"] != "PERUPETRO"][["N", "TEMA", "CAPA", "GRUPO", "GEOMETRIA", "FUENTE"]])

# ── Propuesta automática (revísala) ─────────────────────────────────────────
con_geo = [c for c in capas_pp if c["GEOMETRIA"] and not c.get("ERROR")]
poligonos = [c for c in con_geo if "Polygon" in c["GEOMETRIA"]]
es_lote = lambda c: re.search(r"LOTE|CONTRAT|BLOQUE|ASIGNAC|EXPLOTAC|EXPLORAC", texto(c)) and not re.search(r"CUENCA|SISMIC", texto(c))
def mejor(cands, patron, extra=r"SERNANP|CULTURA|MINCU|GOB.PE|MINEM|OSINERG"):
    puntaje = lambda c: (bool(re.search(patron, texto(c))) * 2 + bool(re.search(extra, texto(c))))
    cands = sorted(cands, key=puntaje, reverse=True)
    return cands[0]["N"] if cands and puntaje(cands[0]) >= 2 else None

OSM = "OpenStreetMap (respaldo)"; NINGUNA = "(ninguna)"
_l = {"layout": w.Layout(width="95%")}
sel_capas = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in con_geo],
                             value=[c["N"] for c in con_geo if not c.get("WFS")],      # las del geoportal: solo las que se usan
                             rows=min(12, max(3, len(con_geo))), description="Guardar extra", **_l)
lotes_ext = [c for c in capas_ext if c["TEMA"] == "LOTES"]
_prio = {"LOTE_CONTRATO": 0, "LOTES_CONTRATO": 1}            # contratos vigentes primero
_wl = sorted([c for c in poligonos if c.get("WFS") and norm_txt(c["WFS"].split(":")[-1]) in _prio],
             key=lambda c: _prio[norm_txt(c["WFS"].split(":")[-1])])
_lv = [c["N"] for c in _wl] or [c["N"] for c in poligonos if not c.get("WFS") and es_lote(c)]
_m = mejor(lotes_ext, r"LOTE", r"PERUPETRO|MINEM|OSINERG|GOB.PE|INGEMMET|GEOCATMIN")
if not _lv:                                   # PERUPETRO no da lotes: mejor candidata externa
    _lv = [_m] if _m is not None else []
    print("\n⚠️ PERUPETRO no entregó polígonos de lotes; propongo:",
          POR_N_[_lv[0]]["CAPA"] + " · " + POR_N_[_lv[0]]["FUENTE"] if _lv else "ninguna (agrega una URL en URLS_EXTRA)")
elif _m is not None:                          # complemento: lotes antiguos que ya no tienen contrato vigente
    _lv.append(_m)
    print(f"\nℹ️ Lotes: geoportal de PERUPETRO (vigentes) + {POR_N_[_m]['CAPA']} ({POR_N_[_m]['FUENTE']}) para lotes antiguos; "
          "si un lote está en ambas, manda el de PERUPETRO.")
sel_lotes = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in poligonos + lotes_ext], value=_lv,
                             rows=min(10, max(3, len(poligonos) + len(lotes_ext))), description="Son lotes", **_l)
_cu = mejor(poligonos, r"CUENCA")
dd_cuenca = w.Dropdown(options=[(NINGUNA, None)] + [(etiqueta(c), c["N"]) for c in poligonos], value=_cu,
                       description="Cuencas", **_l)
def dd_tema(tema, patron, respaldo):
    pp = [c for c in con_geo if any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema])]
    ex = [c for c in capas_ext if c["TEMA"] == tema]
    ops = [(etiqueta(c), c["N"]) for c in pp + ex] + [(respaldo, respaldo)]
    val = mejor(pp, patron) if mejor(pp, patron) is not None else mejor(ex, patron)
    return w.Dropdown(options=ops, value=val if val is not None else respaldo, description=tema, **_l)
dd_anp = dd_tema("ANP", r"AREA(S)? NATURAL|PROTEGID|\bANP\b", OSM)
dd_ccnn = dd_tema("CCNN", r"COMUNIDAD(ES)?.(NATIVA|AMAZONICA)|CC\.?NN", NINGUNA)
if dd_ccnn.value == NINGUNA:
    dd_ccnn.value = (lambda v: v if v is not None else NINGUNA)(
        mejor([c for c in con_geo + capas_ext if any(g in c["GEOMETRIA"] for g in GEOM_TEMA["CCNN"])], r"COMUNIDAD|PUEBLOS? INDIGENA"))
dd_onp = dd_tema("OLEODUCTO", r"NORPERUANO|\bONP\b", OSM)
if dd_onp.value == OSM:                       # ninguna se llama "Norperuano": cualquier oleoducto
    dd_onp.value = (lambda v: v if v is not None else OSM)(
        mejor([c for c in con_geo + capas_ext if "Polyline" in c["GEOMETRIA"]], r"OLEODUCTO|DUCTO"))
txt_caudal = w.IntText(value=5000, description="Río grande: caudal medio ≥ (m³/s)", style={"description_width": "260px"})
display(w.VBox([w.HTML("<b>Capas de PERUPETRO a descargar</b> (Ctrl/Cmd + clic para cambiar la selección)"), sel_capas,
                w.HTML("<b>¿Cuáles son polígonos de lotes?</b>"), sel_lotes,
                w.HTML("<b>Fuentes para los indicadores</b>"), dd_cuenca, dd_onp, dd_anp, dd_ccnn, txt_caudal]))

def _nom(v):
    return v if isinstance(v, str) or v is None else f"[{v}] {POR_N_[v]['CAPA']} · {POR_N_[v]['FUENTE']}"
print("\n✅ Propuesta que usará el Bloque 3 (cámbiala en los selectores o con los números N al inicio del Bloque 3):")
print("   Lotes:     ", "; ".join(_nom(v) for v in sel_lotes.value) or "NINGUNA ⚠️")
print("   Cuencas:   ", _nom(dd_cuenca.value) or "(ninguna)")
print("   Oleoducto: ", _nom(dd_onp.value))
print("   ANP:       ", _nom(dd_anp.value))
print("   Com. nat.: ", _nom(dd_ccnn.value))

cat_ruta = os.path.join(ruta, "CONTROL PP mapa capas.csv")
cat.assign(FECHA_CONSULTA=date.today().strftime("%d/%m/%Y")).to_csv(cat_ruta, index=False, encoding="utf-8-sig")
subir(cat_ruta, DRIVE_FOLDER_CONTROL_ID, control_drive)
print("☁️  Catálogo de capas guardado en CONTROL. Revisa la selección y ejecuta el Bloque 3.")

In [ ]:
# ═══ BLOQUE 3 · DESCARGA Y VARIABLES POR LOTE ═══
# Lote de producción → polígono(s) del mapa del mismo campo (se suman si son varios)
LOTES_ALIAS_GEOMETRIA = {"1-AB": ["192"], "Z-69": ["Z-2B"], "Z-2B": ["Z-69"], "31-B/D": ["31-B", "31-D"]}
# Lotes que se integraron a otro sin compartir polígono (nota del anual 1996 de PERUPETRO)
SUCESION_MANUAL = {"LOB/EA/LO": "X (Folche y Cruz al VI), desde 24/07/1996"}
# Si los selectores no se ven o quieres otra fuente: números N de las tablas del Bloque 2 (vacío/None = propuesta)
N_LOTES = []          # p. ej. [14, 15]
N_CUENCAS = None
N_OLEODUCTO = None    # o "OpenStreetMap (respaldo)"
N_ANP = None
N_CCNN = None
SEL_LOTES = list(N_LOTES) or list(sel_lotes.value)
V_CUENCA = N_CUENCAS if N_CUENCAS is not None else dd_cuenca.value
V_ONP = N_OLEODUCTO if N_OLEODUCTO is not None else dd_onp.value
V_ANP = N_ANP if N_ANP is not None else dd_anp.value
V_CCNN = N_CCNN if N_CCNN is not None else dd_ccnn.value
POR_N = {c["N"]: c for c in capas}
log = []

def guardar_geojson(gdf, nombre):
    p = os.path.join(ruta, nombre)
    gdf.to_file(p, driver="GeoJSON")
    subir(p, MAPA_FOLDER_ID, mapa_drive, mime="application/geo+json")

def capa_gdf(n, cache={}):
    if n in cache:
        return cache[n]
    c = POR_N[n]
    print(f"⬇️  {c['CAPA']} …", end=" ")
    try:
        g = descargar_capa(c)
        print(f"{len(g)} entidades")
        guardar_geojson(g, f"{'PP' if c['FUENTE'] == 'PERUPETRO' else 'EXT'}_{slug(c['CAPA'])}.geojson")
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": len(g), "ESTADO": "OK"})
    except Exception as e:
        print(f"❌ {e}"); g = None
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
    cache[n] = g
    return g

for n in sel_capas.value:
    capa_gdf(n)

# ── 1. Lotes ─────────────────────────────────────────────────────────────────
partes = []
for n in SEL_LOTES:
    g = capa_gdf(n)
    if g is None or g.empty:
        continue
    f_lote = campo_lote(g)
    if not f_lote:
        print(f"⚠️ {POR_N[n]['CAPA']}: no encontré el campo con el nombre del lote. Campos: {list(g.columns)}"); continue
    f_op = campo(g, r"OPERADOR|CONTRATISTA|EMPRESA|COMPANIA|TITULAR|CIA_OPER|^CIA")
    f_est = campo(g, r"ESTADO|SITUACION|FASE|ETAPA|^CONTRATO$")
    f_cu = campo(g, r"CUENCA")
    f_tipo = campo(g, r"TIPO_CONTR|MODALIDAD")
    f_fec = campo(g, r"F_SUSCRIP|FECHA_SUSC|SUSCRIP")
    txt = lambda f: g[f].astype(str).replace({"None": "", "nan": ""}) if f else ""
    p_ = gpd.GeoDataFrame({
        "LOTE": g[f_lote].map(lote_clave), "LOTE_MAPA": g[f_lote].astype(str),
        "OPERADOR_MAPA": txt(f_op), "ESTADO_MAPA": txt(f_est), "TIPO_CONTRATO": txt(f_tipo), "F_SUSCRIPCION": g[f_fec].map(fecha_dmy) if f_fec else "",
        "CUENCA_ATRIBUTO": txt(f_cu), "CAPA": POR_N[n]["CAPA"], "FUENTE_POLIGONO": POR_N[n]["FUENTE"],
        "_PRIORIDAD": len(partes)}, geometry=g.geometry, crs=4326)
    p_ = p_[p_.geometry.notna() & ~p_.geometry.is_empty & p_["LOTE"].ne("")]
    p_["geometry"] = p_.geometry.buffer(0)
    partes.append(p_.dissolve("LOTE", aggfunc="first").reset_index())     # una fila por lote y fuente
if not partes:
    raise RuntimeError("No hay polígonos de lotes: el servidor de PERUPETRO no responde y no se eligió otra fuente. "
                       "Elige una capa de lotes en el Bloque 2 (o N_LOTES arriba), agrega una URL en URLS_EXTRA, "
                       "o usa el shapefile de PERUPETRO (Banco de Datos / promocion@perupetro.com.pe).")
lotes = pd.concat(partes, ignore_index=True).sort_values("_PRIORIDAD")
lotes = lotes.drop_duplicates("LOTE", keep="first").reset_index(drop=True)                            # manda la 1.ª fuente

print("   Polígonos por fuente: " + " · ".join(f"{k}: {v}" for k, v in lotes["CAPA"].value_counts().items()))
lotes["GEOMETRIA_DE"] = lotes["LOTE"]

# Producción (último CSV del notebook de producción): para ubicar los lotes con otro nombre en el mapa
def _cobertura(nombre):
    """(-año inicio, año fin) de «…_<ini>-<fin>.csv»: se prefiere el archivo que cubre MÁS años
    (una corrida rápida «2026-2026» no debe reemplazar a la serie «1994-2026»)."""
    a = re.findall(r"(\d{4})-(\d{4})", nombre)
    return (-int(a[-1][0]), int(a[-1][1])) if a else (0, 0)

prod_csv = sorted([k for k in datos_drive if k.startswith("PP_produccion_lote_mes_") and k.endswith(".csv")], key=_cobertura)
prod = None
if prod_csv:
    p = os.path.join(ruta, prod_csv[-1]); bajar_de_drive(datos_drive[prod_csv[-1]], p)
    prod = pd.read_csv(p, encoding="utf-8-sig")
    prod["OPERADOR"] = prod["OPERADOR"].map(norm_operador)    # por si el CSV es anterior a la homologación
# Gas natural fiscalizado (MPC), del mismo notebook de producción (1994–hoy)
gas_csv = sorted([k for k in datos_drive if k.startswith("PP_produccion_gas_lote_mes_") and k.endswith(".csv")], key=_cobertura)
gas = None
if gas_csv:
    p = os.path.join(ruta, gas_csv[-1]); bajar_de_drive(datos_drive[gas_csv[-1]], p)
    gas = pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str})
    gas["OPERADOR"] = gas["OPERADOR"].map(norm_operador)
# Producción diaria (notebook de producción, sección 8): para la vista «Diaria» del panel
diaria = None
if "PP_produccion_diaria.csv" in datos_drive:
    p = os.path.join(ruta, "PP_produccion_diaria.csv"); bajar_de_drive(datos_drive["PP_produccion_diaria.csv"], p)
    diaria = pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_ORIGINAL": str})
# Un polígono de una fuente secundaria que ya está cubierto (> 50 % de su área) por la fuente principal es el mismo
# lote con otro nombre o dividido en sectores (XIII-A/B/C ⊂ XIII; Z-2B A..D ⊂ Z-69): se descarta, salvo que tenga
# producción propia (entonces es un lote antiguo distinto y se conserva).
_pri = lotes[lotes["_PRIORIDAD"] == lotes["_PRIORIDAD"].min()]
if len(_pri) and (lotes["_PRIORIDAD"] > lotes["_PRIORIDAD"].min()).any():
    _u = _pri.to_crs(CRS_AREA).geometry.union_all() if hasattr(_pri.geometry, "union_all") else _pri.to_crs(CRS_AREA).geometry.unary_union
    _sec = lotes["_PRIORIDAD"] > lotes["_PRIORIDAD"].min()
    _g = lotes.loc[_sec].to_crs(CRS_AREA).geometry
    _cubierto = (_g.intersection(_u).area / _g.area.where(_g.area > 0)).fillna(0) > 0.5
    _con_prod = set(prod["LOTE"].astype(str)) if prod is not None else set()
    _fuera = [i for i in _cubierto[_cubierto].index if lotes.loc[i, "LOTE"] not in _con_prod]
    if len(_fuera):
        print(f"   Polígonos secundarios ya cubiertos por la fuente principal (se descartan): {', '.join(lotes.loc[_fuera, 'LOTE'])}")
        lotes = lotes.drop(index=_fuera).reset_index(drop=True)
lotes = lotes.drop(columns="_PRIORIDAD")
en_mapa = set(lotes["LOTE"])
def poligonos_de(lote):
    """Lotes del mapa que corresponden a un lote de producción: alias, sin sufijo (T)/(TT) o partes de 'VII/VI'."""
    base = re.sub(r"\(T+\)$", "", lote)
    for cand in (LOTES_ALIAS_GEOMETRIA.get(lote), [base] if base != lote else None,
                 LOTES_ALIAS_GEOMETRIA.get(base), base.split("/") if "/" in base else None):
        if cand and all(c in en_mapa for c in cand):
            return cand
    if cand := [c for c in (LOTES_ALIAS_GEOMETRIA.get(lote) or LOTES_ALIAS_GEOMETRIA.get(base) or []) if c in en_mapa]:
        return cand                                       # al menos una parte está en el mapa
    for nombre in [base] + (LOTES_ALIAS_GEOMETRIA.get(base) or []):
        # el mapa divide el lote en sectores: Z-2B → Z-2B A/B/C/D, XIII → XIII-A/B/C
        if cand := sorted(c for c in en_mapa if re.fullmatch(re.escape(nombre) + r"-?[A-Z]", c)):
            return cand
    return None
extra = []
for lote in sorted(set(prod["LOTE"].astype(str)) if prod is not None else set(LOTES_ALIAS_GEOMETRIA)):
    if lote in en_mapa:
        continue
    cand = poligonos_de(lote)
    if cand:
        f = lotes[lotes["LOTE"].isin(cand)]
        extra.append({**f.iloc[0].drop("geometry").to_dict(), "LOTE": lote, "GEOMETRIA_DE": "+".join(cand),
                      "geometry": f.geometry.union_all() if hasattr(f.geometry, "union_all") else f.geometry.unary_union})
if extra:
    lotes = pd.concat([lotes, gpd.GeoDataFrame(extra, geometry="geometry", crs=4326)], ignore_index=True)
lotes = gpd.GeoDataFrame(lotes, geometry="geometry", crs=4326)
print(f"\n🧩 Lotes con polígono: {len(en_mapa)} en el mapa + {len(extra)} ubicados por equivalencia "
      f"({', '.join(e['LOTE'] + '→' + e['GEOMETRIA_DE'] for e in extra) or 'ninguno'})")

# ── 2. Capas externas ────────────────────────────────────────────────────────
def externa(nombre, fn):
    print(f"⬇️  {nombre} …", end=" ")
    try:
        g = fn(); print(f"{len(g)} entidades")
        guardar_geojson(g, f"EXT_{slug(nombre)}.geojson")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": len(g), "ESTADO": "OK"})
        return g
    except Exception as e:
        print(f"❌ {e}")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
        return None

def elegida(v, respaldo_fn, nombre):
    if v in (None, NINGUNA):
        return None, None
    if v == OSM:
        return externa(f"{nombre} OSM", respaldo_fn), "NOMBRE"
    g = capa_gdf(v)
    return g, (campo(g, r"^(NOMBRE|NOM_ANP|ANP_NOMB|NOMB_CCNN|NOM_COMUNI|NAME)$", r"NOMB|NOMBRE|NAME|DENOMIN") if g is not None else None)

deps = externa("Departamentos geoBoundaries", lambda: departamentos(ruta))
# Umbrales amazónicos (m³/s): 1.000 afluentes navegables, 5.000 ríos troncales, 20.000 Amazonas / Marañón y Ucayali bajos
CAUDAL = int(txt_caudal.value or 5000)
UMBRALES = sorted({1000, 5000, 20000, CAUDAL})
rios_q = externa("Rios HydroRIVERS", lambda: rios_hydrorivers(ruta, min(UMBRALES)))
if rios_q is not None and deps is not None:          # solo ríos dentro de Perú
    rios_q = gpd.clip(rios_q, deps.dissolve()[["geometry"]]).explode(index_parts=False)
    rios_q = rios_q[rios_q.geom_type.isin(["LineString", "MultiLineString"])]
rios_n = externa("Rios OSM con nombre", rios_osm_nombres)
if rios_n is None:
    rios_n = externa("Rios Natural Earth", lambda: rios_natural_earth(ruta))
onp, onp_n = elegida(V_ONP, oleoducto_osm, "Oleoducto")
anp, anp_n = elegida(V_ANP, anp_osm, "ANP")
ccnn, ccnn_n = elegida(V_CCNN, None, "Comunidades nativas")
cuencas = capa_gdf(V_CUENCA) if V_CUENCA is not None else None
cuenca_n = campo(cuencas, r"CUENCA", r"NOMB|NOMBRE|NAME") if cuencas is not None else None

# ── 3. Indicadores ───────────────────────────────────────────────────────────
geo = lotes.copy()
geo["AREA_KM2"] = (geo.to_crs(CRS_AREA).geometry.area / 1e6).round(1)
rp = geo.geometry.representative_point()
geo["LAT"], geo["LON"] = rp.y.round(5), rp.x.round(5)
if deps is not None:
    geo = geo.join(reparto_por_poligonos(geo, deps, "DEPARTAMENTO", "DEPARTAMENTO"))
    sin = geo["DEPARTAMENTO"].isna()
    # zócalo: lote mayormente en el mar (la línea de costa generalizada puede tocar una franja de tierra)
    geo["ZOCALO"] = (sin | (geo["DEPARTAMENTO_PCT_TOTAL"].fillna(0) < 50)).map({True: "SI", False: "NO"})
    geo.loc[(geo["ZOCALO"] == "SI") & ~sin, "DEPARTAMENTO_DETALLE"] = (
        "ZÓCALO; en tierra: " + geo.loc[(geo["ZOCALO"] == "SI") & ~sin, "DEPARTAMENTO_DETALLE"].astype(str))
    if sin.any():                                    # lotes en el mar: región costera más cercana
        d, n = distancia_km(geo[sin], deps, "DEPARTAMENTO")
        geo.loc[sin, "DEPARTAMENTO"] = n.values
        geo.loc[sin, "DEPARTAMENTO_DETALLE"] = "ZÓCALO frente a " + n.astype(str).values
if cuencas is not None and cuenca_n:
    geo = geo.join(reparto_por_poligonos(geo, cuencas, cuenca_n, "CUENCA")[["CUENCA", "CUENCA_PCT"]])
else:
    geo["CUENCA"] = geo["CUENCA_ATRIBUTO"]
if onp is not None:
    geo["DIST_OLEODUCTO_KM"], _ = distancia_km(geo, onp)
geo["DIST_IQUITOS_KM"] = (geo.to_crs(CRS_DIST).distance(gpd.GeoSeries([IQUITOS], crs=4326).to_crs(CRS_DIST).iloc[0]) / 1000).round(1)
# distancias a ríos desde el punto interior del lote (desde el borde casi todos los lotes amazónicos darían 0)
pts = gpd.GeoDataFrame(geometry=geo.geometry.representative_point(), crs=4326, index=geo.index)
if rios_q is not None:
    for u in UMBRALES:
        geo[f"DIST_RIO_{u}_KM"], _ = distancia_km(pts, rios_q[rios_q["DIS_AV_CMS"] >= u])
    geo["DIST_RIO_GRANDE_KM"] = geo[f"DIST_RIO_{CAUDAL}_KM"]
if rios_n is not None:
    cand = rios_n
    if rios_q is not None:                               # nombre del río grande más cercano (no de una quebrada)
        grandes = rios_q[rios_q["DIS_AV_CMS"] >= CAUDAL].to_crs(CRS_DIST).buffer(1500)
        zona = gpd.GeoDataFrame(geometry=[grandes.union_all() if hasattr(grandes, "union_all") else grandes.unary_union], crs=CRS_DIST)
        cand = gpd.sjoin(rios_n.to_crs(CRS_DIST), zona, predicate="intersects").drop(columns="index_right").to_crs(4326)
        cand = cand if len(cand) else rios_n
    _, geo["RIO_CERCANO"] = distancia_km(pts, cand, "RIO")
if anp is not None and anp_n:
    a = reparto_por_poligonos(geo, anp, anp_n, "ANP")
    geo["ANP_PCT"] = a.get("ANP_PCT_TOTAL"); geo["ANP_DETALLE"] = a.get("ANP_DETALLE")
    geo["ANP_PCT"] = geo["ANP_PCT"].fillna(0); geo["ANP_DETALLE"] = geo["ANP_DETALLE"].fillna("")
if ccnn is not None and ccnn_n:
    geo = geo.join(comunidades(geo, ccnn, ccnn_n, 5))

# ── 4. Producción: indicadores por lote y por campo ─────────────────────────
def enteros(df):
    """Columnas enteras sin decimales en el CSV (1994 y no 1994.0), aunque tengan vacíos."""
    for c in ["BPD_ACTUAL", "PICO_AÑO", "PICO_BPD", "MESES_SIN_PRODUCCION", "PARALIZACION_MAX_MESES",
              "CCNN_DENTRO_N", "CCNN_5KM_N", "DEPARTAMENTO_N"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")
    return df

COLS_IND = ["ESTADO_PRODUCCION", "PRODUCTO_PRINCIPAL", "PRIMERA_PRODUCCION", "ULTIMA_PRODUCCION", "BPD_ACTUAL",
            "PICO_AÑO", "PICO_BPD", "DECLINACION_ANUAL_5A_PCT", "DECLINACION_PERIODO", "MESES_SIN_PRODUCCION", "PARALIZACION_MAX_MESES",
            "PARALIZACION_MAX_DESDE", "OPERADORES"]
ind_lote, ind_campo, CAMPO_DE = {}, {}, {}
if prod is not None:
    # campo = lotes que comparten polígono (1-AB/192; 31-B, 31-D y 31-B/D; Z-2B y Z-69…)
    hist = (prod.drop_duplicates("LOTE").set_index("LOTE")["LOTE_HISTORICO"].astype(str).to_dict()
            if "LOTE_HISTORICO" in prod else {})
    for lote, grupo in campos(geo).items():
        CAMPO_DE[lote] = " + ".join(sorted({hist.get(l, l) for l in grupo if l in hist} or set(grupo)))
    for lote in prod["LOTE"].astype(str).unique():
        CAMPO_DE.setdefault(lote, hist.get(lote, lote))
    prod["LOTE"] = prod["LOTE"].astype(str)
    prod["CAMPO"] = prod["LOTE"].map(CAMPO_DE)
    ind_lote = indicadores_produccion(prod, "LOTE", grupo="CAMPO")
    ind_campo = indicadores_produccion(prod, "CAMPO")
    # un contrato que terminó y siguió con otro en el mismo campo (1-AB → 192) no está "paralizado": fue sucedido
    def _t(mmaaaa):
        mm, aa = str(mmaaaa).lstrip("≤").split("/"); return int(aa) * 12 + int(mm)
    for l, d in ind_lote.items():
        if d.get("ESTADO_PRODUCCION") != "PARALIZADO":
            continue
        despues = [o for o, e in ind_lote.items() if o != l and CAMPO_DE.get(o) == CAMPO_DE.get(l)
                   and e.get("ULTIMA_PRODUCCION") and _t(e["ULTIMA_PRODUCCION"]) > _t(d["ULTIMA_PRODUCCION"])]
        if not despues and l in SUCESION_MANUAL:
            despues = [SUCESION_MANUAL[l]]
        if despues:
            d["ESTADO_PRODUCCION"] = f"SUCEDIDO POR {', '.join(sorted(despues))}"
            d.update({"PARALIZACION_MAX_MESES": None, "PARALIZACION_MAX_DESDE": None, "MESES_SIN_PRODUCCION": None})
    geo["CAMPO"] = geo["LOTE"].map(CAMPO_DE)
    geo["PETROLEO_ACUM_MMBBL"] = geo["LOTE"].map(lambda l: ind_lote.get(l, {}).get("PETROLEO_ACUM_MMBBL", 0.0))
    geo["PETROLEO_ACUM_CAMPO_MMBBL"] = geo["CAMPO"].map(lambda k: ind_campo.get(k, {}).get("PETROLEO_ACUM_MMBBL", 0.0))
    if gas is not None and len(gas):
        _h = gas["LOTE_HISTORICO"].astype(str) if "LOTE_HISTORICO" in gas else gas["LOTE"]
        gas["CAMPO"] = gas["LOTE"].map(CAMPO_DE).fillna(_h)
        _acum = gas.groupby("CAMPO")["MPC"].sum() / 1e6               # miles de millones de pies cúbicos (BCF)
        geo["GAS_ACUM_CAMPO_BCF"] = geo["CAMPO"].map(_acum).round(2)
    for c in COLS_IND:                                  # indicadores del campo al que pertenece el polígono
        geo[c] = geo["CAMPO"].map(lambda k, c=c: ind_campo.get(k, {}).get(c))
    geo["ESTADO_PRODUCCION"] = geo["ESTADO_PRODUCCION"].fillna("SIN PRODUCCIÓN REGISTRADA")

    tind = pd.DataFrame([{"LOTE": l, "CAMPO": CAMPO_DE.get(l, l),
                          "DEPARTAMENTO": dict(zip(geo["LOTE"], geo.get("DEPARTAMENTO", pd.Series(dtype=str)))).get(l, ""),
                          "TIENE_POLIGONO": "SI" if l in set(geo["LOTE"]) else "NO",
                          **{k: v for k, v in d.items() if not k.startswith("_")}} for l, d in ind_lote.items()])
    if "DEPARTAMENTO" in prod:                          # lotes sin polígono: departamento de la base de producción
        dep_prod = prod[prod["DEPARTAMENTO"].fillna("") != ""].groupby("LOTE")["DEPARTAMENTO"].agg(lambda s: s.mode().iloc[0])
        tind["DEPARTAMENTO"] = tind["DEPARTAMENTO"].replace("", None).fillna(tind["LOTE"].map(dep_prod)).fillna("")
    tind = enteros(tind).sort_values(["DEPARTAMENTO", "PETROLEO_ACUM_MMBBL"], ascending=[True, False])
    tind.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
    p = os.path.join(ruta, "PP_LOTES_INDICADORES.csv"); tind.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)

    falta = sorted(set(prod["LOTE"]) - set(geo["LOTE"]))
    print(f"\n🔗 Producción leída de {prod_csv[-1]} · indicadores de {len(ind_lote)} lotes y {len(ind_campo)} campos "
          f"→ DATA/PP_LOTES_INDICADORES.csv")
    print(f"   Lotes con producción y sin polígono ({len(falta)}): {', '.join(falta) if falta else 'ninguno'}")
    usados = {p for e in extra for p in e["GEOMETRIA_DE"].split("+")}
    sin_prod = sorted(en_mapa - set(prod["LOTE"]) - usados)
    print(f"   Lotes del mapa sin producción ({len(sin_prod)}): {', '.join(sin_prod)}")
    log.append({"CAPA": "LOTES SIN POLIGONO", "URL": "", "FUENTE": prod_csv[-1], "ENTIDADES": len(falta),
                "ESTADO": ", ".join(falta)[:500]})
else:
    geo["ESTADO_PRODUCCION"] = "SIN DATOS DE PRODUCCIÓN"
    print("\nℹ️ No encontré PP_produccion_lote_mes_*.csv en DATA: corre antes el notebook de producción para el cruce.")

# ── 5. Guardar ───────────────────────────────────────────────────────────────
cols = [c for c in ["LOTE", "GEOMETRIA_DE", "LOTE_MAPA", "OPERADOR_MAPA", "ESTADO_MAPA", "TIPO_CONTRATO", "F_SUSCRIPCION",
                    "CAPA", "FUENTE_POLIGONO", "DEPARTAMENTO",
                    "DEPARTAMENTO_PCT", "DEPARTAMENTO_DETALLE", "DEPARTAMENTO_N", "ZOCALO", "CUENCA", "CUENCA_PCT",
                    "AREA_KM2", "LAT", "LON", "DIST_OLEODUCTO_KM", "DIST_IQUITOS_KM", "DIST_RIO_GRANDE_KM"] + [f"DIST_RIO_{u}_KM" for u in UMBRALES] + ["RIO_CERCANO",
                    "ANP_PCT", "ANP_DETALLE", "CCNN_DENTRO_N", "CCNN_5KM_N", "CCNN_DENTRO", "CAMPO", "PETROLEO_ACUM_MMBBL",
                    "PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF"] + COLS_IND
        if c in geo.columns]
tabla = enteros(pd.DataFrame(geo[cols])).sort_values("LOTE")
tabla.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
p = os.path.join(ruta, "PP_LOTES_GEO.csv"); tabla.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_ID, datos_drive)
guardar_geojson(geo[cols + ["geometry"]], "PP_lotes_indicadores.geojson")
pc = os.path.join(ruta, "CONTROL PP mapa descarga.csv")
pd.DataFrame(log).assign(FECHA=date.today().strftime("%d/%m/%Y")).to_csv(pc, index=False, encoding="utf-8-sig")
subir(pc, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️  DATA/PP_LOTES_GEO.csv ({len(tabla)} lotes) · DATA/MAPA/PP_lotes_indicadores.geojson · CONTROL actualizado")

if "DEPARTAMENTO" in tabla:
    print("\n📍 Lotes por departamento:")
    display(tabla.groupby("DEPARTAMENTO").agg(LOTES=("LOTE", "count"), AREA_KM2=("AREA_KM2", "sum")).sort_values("LOTES", ascending=False))
    lor = tabla[tabla["DEPARTAMENTO"] == "LORETO"]
    if len(lor):
        print("\n🛢️ Loreto (indicadores del campo):")
        ver = [c for c in ["LOTE", "CAMPO", "ESTADO_PRODUCCION", "BPD_ACTUAL", "PICO_AÑO", "PICO_BPD", "DECLINACION_ANUAL_5A_PCT",
                           "PARALIZACION_MAX_MESES", "PETROLEO_ACUM_CAMPO_MMBBL", "DIST_OLEODUCTO_KM", "DIST_RIO_GRANDE_KM",
                           "ANP_PCT", "CCNN_DENTRO_N"] if c in lor]
        display(lor[ver].sort_values("PETROLEO_ACUM_CAMPO_MMBBL" if "PETROLEO_ACUM_CAMPO_MMBBL" in lor else "LOTE", ascending=False))

In [ ]:
# ═══ BLOQUE 4 · MAPA INTERACTIVO ═══
import folium, math, json, html as _h, shapely, unicodedata
from folium.plugins import MarkerCluster
m = folium.Map(location=[-4.8, -75.5], zoom_start=6, tiles=None, control_scale=True)
# Fondos de Esri: los mosaicos de OpenStreetMap bloquean las vistas embebidas de Colab ("403 Access blocked")
_ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
folium.TileLayer(_ESRI.format("World_Topo_Map"), attr="Esri, HERE, Garmin, USGS, NGA", name="Topográfico (Esri)").add_to(m)
folium.TileLayer(_ESRI.format("World_Imagery"), attr="Esri, Maxar, Earthstar Geographics", name="Satélite (Esri)", show=False).add_to(m)

def simpl(g, tol=0.002):
    # Simplifica y redondea a 5 decimales (~1 m): el mapa queda liviano y abre en cualquier computadora
    g = g.copy()
    g["geometry"] = g.geometry.simplify(tol, preserve_topology=True).map(lambda x: shapely.set_precision(x, 1e-5) if x is not None else x)
    return g[g.geometry.notna() & ~g.geometry.is_empty]

ANIO_INI, ANIO_FIN = (int(prod["AÑO"].min()), int(prod["AÑO"].max())) if prod is not None else (None, None)
COLOR_ESTADO = {"EN PRODUCCIÓN": "#2e7d32", "PARALIZADO": "#ef6c00", "SIN PRODUCCIÓN REGISTRADA": "#9e9e9e",
                "SIN DATOS DE PRODUCCIÓN": "#9e9e9e"}
ETIQUETAS = [("CAMPO", "Campo", ""), ("OPERADOR_MAPA", "Operador (contrato vigente)", ""), ("ESTADO_MAPA", "Fase del contrato", ""),
             ("TIPO_CONTRATO", "Tipo de contrato", ""), ("F_SUSCRIPCION", "Suscripción", ""),
             ("DEPARTAMENTO_DETALLE", "Departamento", ""), ("AREA_KM2", "Área", "km²"),
             ("PRIMERA_PRODUCCION", "Primera producción", ""),
             ("ULTIMA_PRODUCCION", "Última producción", ""), ("PICO_AÑO", "Año pico", ""),
             ("PICO_BPD", "Producción en el pico", "bpd"), ("DECLINACION_ANUAL_5A_PCT", "Variación anual (5 años)", "%"),
             ("DECLINACION_PERIODO", "… periodo", ""),
             ("PARALIZACION_MAX_MESES", "Paralización más larga", "meses"), ("PARALIZACION_MAX_DESDE", "… desde", ""),
             ("PETROLEO_ACUM_CAMPO_MMBBL", "Petróleo acumulado del campo", "MM bbl"),
             ("GAS_ACUM_CAMPO_BCF", "Gas natural acumulado del campo", "BCF"), ("OPERADORES", "Operadores", ""),
             ("DIST_OLEODUCTO_KM", "Distancia al oleoducto", "km"),
             ("DIST_RIO_1000_KM", "Distancia a un afluente (≥ 1.000 m³/s)", "km"),
             ("DIST_RIO_GRANDE_KM", "Distancia a un río troncal (≥ 5.000 m³/s)", "km"),
             ("RIO_CERCANO", "Río grande cercano", ""), ("DIST_IQUITOS_KM", "Distancia a Iquitos", "km"),
             ("ANP_DETALLE", "Áreas protegidas", ""), ("CCNN_DENTRO_N", "Comunidades nativas dentro", ""),
             ("CCNN_5KM_N", "Comunidades a menos de 5 km", "")]

ENTEROS = {"BPD_ACTUAL", "PICO_BPD", "PARALIZACION_MAX_MESES", "CCNN_DENTRO_N", "CCNN_5KM_N", "AREA_KM2", "DIST_OLEODUCTO_KM",
           "DIST_RIO_GRANDE_KM", "DIST_RIO_1000_KM", "DIST_IQUITOS_KM"}

def _num(v, dec):                                   # formato peruano: 12.299,5
    return f"{v:,.{dec}f}".replace(",", "§").replace(".", ",").replace("§", ".")

def fmt(c, v, u):
    if v is None or (isinstance(v, float) and math.isnan(v)) or str(v).strip() in ("", "nan", "None", "<NA>"):
        return None
    if c == "PICO_AÑO":
        v = str(int(float(v)))
    elif isinstance(v, (int, float)) and not isinstance(v, bool):
        v = _num(float(v), 0 if c in ENTEROS else (2 if c in ("PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF") else 1))
    return f"{_h.escape(str(v))} {u}".strip()

# ── Series mensuales para el panel de producción (van dentro del HTML: el mapa funciona sin conexión a Drive) ──
PANEL_HTML = r'''<style>
#pp-panel,#pp-abrir,#pp-ley{font:12px/1.4 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:#0b0b0b}
.pp-caja{background:#fcfcfb;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,.28)}
.pp-btn{border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;min-width:26px;height:24px;padding:0 7px;cursor:pointer;
  font:600 13px/20px system-ui,sans-serif;color:#0b0b0b}
.pp-btn:hover{background:#e7e6e0}.pp-btn:focus-visible,.pp-sel:focus-visible{outline:2px solid #2a78d6;outline-offset:1px}
#pp-panel{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);z-index:10001;width:min(900px,calc(100vw - 24px));
  display:none;flex-direction:column;max-height:calc(100vh - 32px);box-shadow:0 8px 32px rgba(0,0,0,.35)}
#pp-fondo{position:fixed;inset:0;z-index:10000;background:rgba(20,20,20,.38);display:none;cursor:pointer}
#pp-cab{display:flex;align-items:center;gap:8px;padding:8px 10px;border-bottom:1px solid #e6e5e0;cursor:default}
#pp-cab b{font-size:14px;flex:1;min-width:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#pp-cuerpo{padding:8px 10px 10px;overflow:auto}
#pp-filtros{display:flex;flex-wrap:wrap;gap:6px 10px;align-items:flex-end;margin-bottom:8px}
#pp-filtros label{display:flex;flex-direction:column;font-size:11px;color:#52514e;gap:2px}
.pp-sel{font:12px system-ui,sans-serif;padding:3px 4px;border:1px solid #c9c8c2;border-radius:5px;background:#fff;color:#0b0b0b;max-width:210px}
#pp-resumen{border-collapse:collapse;margin:2px 0 8px;font-size:12px;width:100%}
#pp-resumen th{font-weight:normal;color:#52514e;text-align:right;padding:2px 8px;border-bottom:1px solid #e6e5e0}
#pp-resumen td{text-align:right;padding:3px 8px;font-variant-numeric:tabular-nums}
#pp-resumen td:first-child,#pp-resumen th:first-child{text-align:left}
#pp-resumen td b{font-weight:600}
#pp-titulo{font-weight:600;margin:2px 0 0}#pp-nota{color:#52514e;font-size:11px;min-height:15px}
#pp-graf{position:relative;width:100%;height:270px;touch-action:none}
#pp-graf svg{display:block;width:100%;height:100%}
#pp-tip{position:absolute;pointer-events:none;background:#fff;border:1px solid #d8d7d1;border-radius:6px;padding:6px 8px;
  box-shadow:0 2px 6px rgba(0,0,0,.18);font-size:12px;display:none;white-space:nowrap;z-index:2}
#pp-tip .v{font-weight:700;font-variant-numeric:tabular-nums}#pp-tip .s{color:#52514e}
#pp-tip i{display:inline-block;width:12px;height:0;border-top:2px solid;vertical-align:middle;margin-right:5px}
#pp-leyenda{display:flex;gap:14px;font-size:12px;color:#52514e;margin-top:2px}
#pp-leyenda span i{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:5px;vertical-align:-1px}
#pp-tabla-c{max-height:220px;overflow:auto;margin-top:8px;border-top:1px solid #e6e5e0;display:none}
#pp-tabla{border-collapse:collapse;width:100%;font-size:12px}
#pp-tabla th{position:sticky;top:0;background:#f3f2ee;font-weight:600;text-align:right;padding:3px 8px}
#pp-tabla td{text-align:right;padding:2px 8px;border-bottom:1px solid #f0efec;font-variant-numeric:tabular-nums}
#pp-tabla td:first-child,#pp-tabla th:first-child,#pp-tabla td:last-child,#pp-tabla th:last-child{text-align:left}
#pp-tabla td:last-child{white-space:normal;color:#52514e}
#pp-abrir{position:fixed;left:10px;bottom:24px;z-index:9999;padding:6px 10px;height:auto;font-size:12px}
#pp-ley{position:fixed;bottom:28px;right:10px;z-index:9999;padding:6px 9px;max-width:240px;font-size:11px;line-height:1.5}
#pp-ley-cab{display:flex;align-items:center;justify-content:space-between;gap:12px}
.pp-mini #pp-cuerpo{display:none}
#pp-panel.pp-mini{top:auto;bottom:24px;transform:translateX(-50%);width:min(520px,calc(100vw - 24px))}
@media (max-width:600px){#pp-graf{height:220px}.pp-sel{max-width:150px}}
</style>
<button id="pp-abrir" class="pp-btn pp-caja" type="button" onclick="ppAbrir()">📊 Producción fiscalizada anual y mensual</button>
<div id="pp-fondo" onclick="ppCerrar()" title="Clic fuera del panel para cerrarlo"></div>
<div id="pp-panel" class="pp-caja" aria-modal="true" role="dialog" aria-label="Producción fiscalizada anual y mensual">
  <div id="pp-cab"><b id="pp-cab-t">Producción</b>
    <button class="pp-btn" type="button" id="pp-min" title="Minimizar" aria-label="Minimizar panel" onclick="ppMin()">–</button>
    <button class="pp-btn" type="button" title="Cerrar" aria-label="Cerrar panel" onclick="ppCerrar()">×</button></div>
  <div id="pp-cuerpo">
    <div id="pp-filtros">
      <label>Serie<select class="pp-sel" id="pp-a"></select></label>
      <label>Comparar con<select class="pp-sel" id="pp-b"></select></label>
      <label>Producto<select class="pp-sel" id="pp-prod"><option value="T">Líquidos totales</option>
        <option value="P">Petróleo</option><option value="L">LGN y condensado</option>
        <option value="G">Gas natural</option></select></label>
      <label>Vista<select class="pp-sel" id="pp-vista"><option value="A">Anual</option><option value="M">Mensual</option></select></label>
      <label>Desde<select class="pp-sel" id="pp-desde"></select></label>
      <label>Hasta<select class="pp-sel" id="pp-hasta"></select></label>
      <label>Unidad<select class="pp-sel" id="pp-u"><option value="bbl">Barriles</option>
        <option value="bpd">Barriles por día</option></select></label>
      <button class="pp-btn" type="button" id="pp-volver" style="display:none" onclick="ppVolver()">↩ Serie anual</button>
      <button class="pp-btn" type="button" id="pp-btab" onclick="ppTabla()">Tabla</button>
      <button class="pp-btn" type="button" onclick="ppCSV()">CSV</button>
    </div>
    <table id="pp-resumen"></table>
    <div id="pp-titulo"></div><div id="pp-nota"></div>
    <div id="pp-graf"><div id="pp-tip"></div></div>
    <div id="pp-leyenda"></div>
    <div id="pp-tabla-c"><table id="pp-tabla"></table></div>
  </div>
</div>
<script>
(function(){
const D=__DATA__;
const MES=["ene","feb","mar","abr","may","jun","jul","ago","set","oct","nov","dic"];
const COL=["#2a78d6","#eb6834"], GRID="#e6e5e0", INK2="#52514e", NS="http://www.w3.org/2000/svg";
const S={}; D.s.forEach(x=>S[x.id]=x);
const A0=D.ini, A1s=D.ini+Math.floor((D.n-1)/12);          // último año de las estadísticas mensuales
const A1=Math.max(A1s,D.dia?new Date(Date.UTC(D.dia.ini[0],D.dia.ini[1]-1,D.dia.ini[2])+(D.dia.n-1)*864e5).getUTCFullYear():0);
const st={a:D.def,b:"",prod:"T",vista:"A",desde:A0,hasta:A1,u:"bbl",tabla:false,prev:null,pila:[],dmes:null};
const DIA=D.dia||null, DIA0=DIA?Date.UTC(DIA.ini[0],DIA.ini[1]-1,DIA.ini[2]):0;
const DA0=DIA?DIA.ini[0]:0, DA1=DIA?new Date(DIA0+(DIA.n-1)*864e5).getUTCFullYear():0;
const pad=v=>(v<10?"0":"")+v;
function mesDiario(x,a,m){const dd=x.dd||{};let b=0,n=0,ult=0;
  for(let i=Math.max(0,Math.round((Date.UTC(a,m,1)-DIA0)/864e5));i<DIA.n;i++){const f=new Date(DIA0+i*864e5);
    if(f.getUTCFullYear()!==a||f.getUTCMonth()!==m)break;
    const p=(dd.p||[])[i]||0,l=(dd.l||[])[i]||0,g=(dd.g||[])[i]||0;
    b+=esGas()?g:st.prod==="P"?p:st.prod==="L"?l:p+l;n++;ult=f.getUTCDate();}
  return n?{b:b,n:n,ult:ult}:null;}
function hayDias(a,m){if(!DIA)return false;const i0=Math.round((Date.UTC(a,m,1)-DIA0)/864e5),i1=Math.round((Date.UTC(a,m+1,1)-DIA0)/864e5);return i1>0&&i0<DIA.n;}
const $=id=>document.getElementById(id);
const PROD={T:"líquidos totales",P:"petróleo",L:"LGN y condensado",G:"gas natural"};
const esGas=()=>st.prod==="G";

function num(v,d){d=d||0;if(v==null||!isFinite(v))return "–";const s=Math.abs(v).toFixed(d).split(".");
  return (v<0?"−":"")+s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
function corto(v){const a=Math.abs(v);if(a>=1e6)return num(v/1e6,a>=1e7?0:1)+" M";if(a>=1e4)return num(v/1e3,0)+" mil";return num(v,0);}
function vol(b){if(esGas())return b>=1e6?num(b/1e6,1)+" BCF":num(b/1e3,0)+" MMPC";   // gas: b en MPC
  return Math.abs(b)>=1e6?num(b/1e6,2)+" MM bbl":num(b,0)+" bbl";}
function dias(t){return new Date(A0+Math.floor(t/12),t%12+1,0).getDate();}
function mensual(x){if(esGas())return x.gs||x.p.map(()=>0);
  return x.p.map((v,i)=>st.prod==="P"?v:st.prod==="L"?x.l[i]:v+x.l[i]);}
function datos(x){
  if(st.vista==="D"){
    const dd=x.dd||{},out=[],g=esGas()?1/1000:1;
    for(let i=0;DIA&&i<DIA.n;i++){const f=new Date(DIA0+i*864e5),a=f.getUTCFullYear(),mm=f.getUTCMonth(),di=f.getUTCDate();
      if(a<st.desde||a>st.hasta||(st.dmes&&(st.dmes.a!==a||st.dmes.m!==mm)))continue;
      const p=(dd.p||[])[i]||0,l=(dd.l||[])[i]||0,gg=(dd.g||[])[i]||0;
      const v=esGas()?gg:st.prod==="P"?p:st.prod==="L"?l:p+l;
      out.push({k:i,lab:pad(di)+"/"+pad(mm+1)+"/"+a,eje:String(di),dia:di,mes:mm,anio:a,bbl:v,d:1,ops:(x.o||{})[a]||"",v:v*g});}
    return out;
  }
  const m=mensual(x),o=x.o||{},out=[];
  for(let a=st.desde;a<=st.hasta;a++){
    if(st.vista==="A"){let b=0,d=0,n=0;for(let k=0;k<12;k++){const t=(a-A0)*12+k;if(t>=D.n)break;b+=m[t];d+=dias(t);n++;}
      if(n)out.push({k:a,lab:a+(n<12?" (ene–"+MES[n-1]+")":""),eje:String(a),bbl:b,d:d,ops:o[a]||"",anio:a});}
    else for(let k=0;k<12;k++){const t=(a-A0)*12+k;
      if(t<D.n)out.push({k:t,lab:MES[k]+" "+a,eje:MES[k],mes:k,bbl:m[t],d:dias(t),ops:o[a]||"",anio:a});
      else if(hayDias(a,k)){const r=mesDiario(x,a,k);   // meses después de la última estadística: suma de los reportes diarios
        if(r)out.push({k:t,lab:MES[k]+" "+a+" (diario"+(r.n<new Date(Date.UTC(a,k+1,0)).getUTCDate()?", días 1–"+r.ult:"")+")",
                       eje:MES[k],mes:k,bbl:r.b,d:r.n,ops:o[a]||"",anio:a,diario:true});}}
  }
  const f=esGas()?1/1000:1;                         // gas: MPC → MMPC
  out.forEach(r=>r.v=(st.u==="bpd"?r.bbl/r.d:r.bbl)*f);return out;
}
function uni(){if(st.vista==="D")return esGas()?"MMPCD":"bpd";                      // un día: volumen = promedio diario
  return esGas()?(st.u==="bpd"?"MMPCD":"MMPC"):(st.u==="bpd"?"bpd":"bbl");}
function el(tag,at,par){const e=document.createElementNS(NS,tag);for(const k in at)e.setAttribute(k,at[k]);if(par)par.appendChild(e);return e;}
function paso(max){if(max<=0)return 1;const r=max/4,e=Math.pow(10,Math.floor(Math.log10(r))),f=r/e;return (f<=1?1:f<=2?2:f<=2.5?2.5:f<=5?5:10)*e;}
function barra(x,y,w,h,r){r=Math.min(r,w/2,h);if(h<=0)return "";
  return `M${x},${y+h}V${y+r}Q${x},${y} ${x+r},${y}H${x+w-r}Q${x+w},${y} ${x+w},${y+r}V${y+h}Z`;}

function opciones(sel,vacia){
  sel.textContent="";if(vacia){const o=document.createElement("option");o.value="";o.textContent="— ninguna —";sel.appendChild(o);}
  const grupos={};D.s.forEach(x=>{(grupos[x.g]=grupos[x.g]||[]).push(x);});
  D.grupos.forEach(g=>{if(!grupos[g])return;const og=document.createElement("optgroup");og.label=g;
    grupos[g].forEach(x=>{const o=document.createElement("option");o.value=x.id;o.textContent=x.n;og.appendChild(o);});sel.appendChild(og);});
}
function anios(){["pp-desde","pp-hasta"].forEach(id=>{const s=$(id);s.textContent="";
  for(let a=A0;a<=A1;a++){const o=document.createElement("option");o.value=a;o.textContent=a;s.appendChild(o);}});}

function series(){const r=[S[st.a]];if(st.b&&S[st.b]&&st.b!==st.a)r.push(S[st.b]);return r;}

function resumen(ss,ds){
  const t=$("pp-resumen");t.textContent="";
  const h=t.insertRow();["Serie","Total del periodo","Promedio diario","Máximo ("+(st.vista==="A"?"año":st.vista==="M"?"mes":"día")+")",
    st.vista==="D"?"Días sin producción":"Meses sin producción"]
    .forEach(c=>{const th=document.createElement("th");th.textContent=c;h.appendChild(th);});
  ss.forEach((x,i)=>{const d=ds[i],tb=d.reduce((s,r)=>s+r.bbl,0),td=d.reduce((s,r)=>s+r.d,0);
    let mx=null;d.forEach(r=>{if(!mx||r.v>mx.v)mx=r;});
    const m=mensual(x);let ceros=0,ini=m.findIndex(v=>v>0);
    if(st.vista==="D")ceros=d.filter(r=>!(r.bbl>0)).length;
    else for(let t=Math.max((st.desde-A0)*12,ini<0?D.n:ini);t<Math.min((st.hasta-A0+1)*12,D.n);t++)if(!(m[t]>0))ceros++;
    const tr=t.insertRow(),c0=tr.insertCell(),sw=document.createElement("i");
    sw.style.cssText=`display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:6px;vertical-align:-1px;background:${COL[i]}`;
    c0.appendChild(sw);c0.appendChild(document.createTextNode(x.n));
    const b=document.createElement("b");b.textContent=vol(tb);tr.insertCell().appendChild(b);
    tr.insertCell().textContent=esGas()?num(td?tb/td/1000:0,1)+" MMPCD":num(td?tb/td:0,0)+" bpd";
    tr.insertCell().textContent=mx&&mx.v>0?num(mx.v,esGas()&&st.u==="bpd"?1:0)+" "+uni()+" · "+mx.lab:"–";
    tr.insertCell().textContent=num(ceros,0);
  });
}

function grafico(ss,ds){
  const box=$("pp-graf"),tip=$("pp-tip");box.querySelectorAll("svg").forEach(s=>s.remove());tip.style.display="none";
  const W=box.clientWidth||700,H=box.clientHeight||270,M={l:52,r:10,t:10,b:26},iw=W-M.l-M.r,ih=H-M.t-M.b;
  const svg=el("svg",{viewBox:`0 0 ${W} ${H}`,role:"img","aria-label":$("pp-titulo").textContent});box.insertBefore(svg,tip);
  const n=ds[0].length;if(!n)return;
  let max=0;ds.forEach(d=>d.forEach(r=>{if(r.v>max)max=r.v;}));
  const p=paso(max),top=Math.max(p,Math.ceil(max/p)*p),y=v=>M.t+ih-(v/top)*ih;
  for(let v=0;v<=top+1e-9;v+=p){el("line",{x1:M.l,x2:W-M.r,y1:y(v),y2:y(v),stroke:v===0?"#b9b8b2":GRID,"stroke-width":1},svg);
    const tx=el("text",{x:M.l-6,y:y(v)+4,"text-anchor":"end","font-size":11,fill:INK2},svg);tx.textContent=corto(v);}
  const bw=iw/n,cx=i=>M.l+bw*(i+.5);
  // etiquetas del eje X sin chocar
  const cada=Math.max(1,Math.ceil(n/Math.max(2,Math.floor(iw/(st.vista==="A"?38:30)))));
  if(st.vista==="D")ds[0].forEach((r,i)=>{
    if(n>45?r.dia===1:i%Math.max(1,Math.ceil(n/Math.floor(iw/26)))===0){
      const tx=el("text",{x:cx(i),y:H-8,"text-anchor":n>45?"start":"middle","font-size":11,fill:INK2},svg);tx.textContent=n>45?MES[r.mes]+(r.mes===0?" "+r.anio:""):r.eje;}});
  else ds[0].forEach((r,i)=>{const enero=st.vista==="M"&&n>24;
    if(enero?(r.mes===0&&((r.anio-st.desde)%Math.max(1,Math.ceil((n/12)/Math.floor(iw/40))))===0):i%cada===0){
      const tx=el("text",{x:cx(i),y:H-8,"text-anchor":"middle","font-size":11,fill:INK2},svg);tx.textContent=enero?String(r.anio):r.eje;}});
  const marcas=el("g",{},svg),cruz=el("line",{y1:M.t,y2:M.t+ih,stroke:"#8a8984","stroke-width":1,visibility:"hidden"},svg);
  const puntos=[];
  if(st.vista==="A"){
    const k=ds.length,g=2,w=Math.max(2,Math.min(46,(bw*.72-g*(k-1))/k));
    ds.forEach((d,j)=>d.forEach((r,i)=>{const x0=cx(i)-(k*w+g*(k-1))/2+j*(w+g),hh=M.t+ih-y(r.v);
      el("path",{d:barra(x0,y(r.v),w,hh,Math.min(4,w/2)),fill:COL[j],"data-i":i},marcas);}));
  } else {
    ds.forEach((d,j)=>{let s="";d.forEach((r,i)=>{s+=(i?"L":"M")+cx(i).toFixed(1)+","+y(r.v).toFixed(1);});
      el("path",{d:s,fill:"none",stroke:COL[j],"stroke-width":2,"stroke-linejoin":"round"},marcas);
      puntos.push(el("circle",{r:4,fill:COL[j],stroke:"#fcfcfb","stroke-width":2,visibility:"hidden"},svg));});
  }
  const hit=el("rect",{x:M.l,y:M.t,width:iw,height:ih,fill:"transparent",style:"cursor:"+(st.vista==="A"||(st.vista==="M"&&DIA)?"pointer":"crosshair"),tabindex:0},svg);
  let actual=-1;
  function mostrar(i){
    if(i<0||i>=n){ocultar();return;}actual=i;const r0=ds[0][i];
    cruz.setAttribute("x1",cx(i));cruz.setAttribute("x2",cx(i));cruz.setAttribute("visibility","visible");
    marcas.querySelectorAll("path[data-i]").forEach(b=>b.setAttribute("opacity",+b.dataset.i===i?1:.55));
    puntos.forEach((c,j)=>{c.setAttribute("cx",cx(i));c.setAttribute("cy",y(ds[j][i].v));c.setAttribute("visibility","visible");});
    tip.textContent="";const t=document.createElement("div");t.className="s";t.textContent=r0.lab;tip.appendChild(t);
    ds.forEach((d,j)=>{const f=document.createElement("div"),k=document.createElement("i"),v=document.createElement("span"),s=document.createElement("span");
      k.style.borderColor=COL[j];v.className="v";v.textContent=num(d[i].v,esGas()&&st.u==="bpd"?1:0)+" "+uni();s.className="s";s.textContent="  "+ss[j].n;
      f.appendChild(k);f.appendChild(v);f.appendChild(s);tip.appendChild(f);
      if(d[i].ops){const o=document.createElement("div");o.className="s";o.style.paddingLeft="17px";o.textContent=d[i].ops;tip.appendChild(o);}});
    const pista=st.vista==="A"?"Clic: ver los meses de "+r0.anio:(st.vista==="M"&&hayDias(r0.anio,r0.mes)?"Clic: ver los días de "+r0.lab:"");
    if(pista){const a=document.createElement("div");a.className="s";a.style.marginTop="3px";a.textContent=pista;tip.appendChild(a);}
    tip.style.display="block";const tw=tip.offsetWidth,x=cx(i)+12;
    tip.style.left=(x+tw>W?cx(i)-tw-12:x)+"px";tip.style.top=M.t+"px";
  }
  function ocultar(){actual=-1;tip.style.display="none";cruz.setAttribute("visibility","hidden");
    puntos.forEach(c=>c.setAttribute("visibility","hidden"));marcas.querySelectorAll("path[data-i]").forEach(b=>b.setAttribute("opacity",1));}
  function indice(ev){const r=svg.getBoundingClientRect(),x=(ev.clientX-r.left)*(W/r.width);return Math.min(n-1,Math.max(0,Math.floor((x-M.l)/bw)));}
  hit.addEventListener("pointermove",ev=>mostrar(indice(ev)));
  hit.addEventListener("pointerleave",ocultar);
  function entrar(r){if(!r)return;if(st.vista==="A")ppMeses(r.anio);else if(st.vista==="M"&&hayDias(r.anio,r.mes))ppDias(r.anio,r.mes);}
  hit.addEventListener("click",ev=>entrar(ds[0][indice(ev)]));
  hit.addEventListener("keydown",ev=>{if(ev.key==="ArrowRight")mostrar(Math.min(n-1,actual+1));else if(ev.key==="ArrowLeft")mostrar(Math.max(0,actual<0?n-1:actual-1));
    else if(ev.key==="Enter"&&actual>=0)entrar(ds[0][actual]);else return;ev.preventDefault();});
  hit.addEventListener("blur",ocultar);
}

function tabla(ss,ds){
  const t=$("pp-tabla");t.textContent="";const h=t.insertRow();
  [st.vista==="A"?"Año":st.vista==="M"?"Mes":"Día"].concat(ss.map(x=>x.n+" ("+uni()+")"),["Operador(es)"]).forEach(c=>{const th=document.createElement("th");th.textContent=c;h.appendChild(th);});
  ds[0].forEach((r,i)=>{const tr=t.insertRow();tr.insertCell().textContent=r.lab;ds.forEach(d=>tr.insertCell().textContent=num(d[i].v,esGas()&&st.u==="bpd"?1:0));
    tr.insertCell().textContent=ss.map(x=>x.o&&x.o[r.anio]||"").filter(Boolean).join(" | ");});
}

function dibujar(){
  if(!S[st.a])st.a=D.def;if(st.desde>st.hasta){const x=st.desde;st.desde=st.hasta;st.hasta=x;}
  ["a","b","prod","vista","desde","hasta","u"].forEach(k=>{$("pp-"+k).value=st[k];});
  const ss=series(),ds=ss.map(datos);
  $("pp-u").options[0].textContent=esGas()?"Millones de pies cúbicos":"Barriles";
  $("pp-u").options[1].textContent=esGas()?"Millones de pies cúbicos por día":"Barriles por día";
  $("pp-cab-t").textContent="Producción fiscalizada · "+ss.map(x=>x.n).join(" vs. ");
  $("pp-titulo").textContent=ss.map(x=>x.n).join(" vs. ")+" · producción fiscalizada de "+PROD[st.prod]+" · "+
    (esGas()?(st.u==="bpd"?"MMPC por día":"millones de pies cúbicos (MMPC)"):(st.u==="bpd"?"barriles por día":"barriles"))+
    (st.vista==="A"?" por año":st.vista==="M"?" por mes":"")+" · "+(st.dmes?MES[st.dmes.m]+" "+st.dmes.a:(st.desde===st.hasta?st.desde:st.desde+"–"+st.hasta));
  if(st.vista==="D")$("pp-titulo").textContent=$("pp-titulo").textContent.replace(" · barriles por día"," · barriles por día (diario)")
    .replace(" · barriles"," · barriles por día (diario)").replace(" · millones de pies cúbicos (MMPC)"," · MMPC por día (diario)");
  const ult=D.n%12?" En las estadísticas mensuales el último año está incompleto (hasta "+MES[(D.n-1)%12]+" "+A1s+").":"";
  let nota;
  $("pp-u").disabled=st.vista==="D";                     // en la vista diaria cada valor ya es «por día»
  if(st.vista==="D")nota=DIA?"Reportes diarios de PERUPETRO (producción fiscalizada del día)."+(esGas()?"":
      " En líquidos varios lotes se fiscalizan por entregas: hay días en 0 y días altos; para comparar usa el promedio diario.")
                          :"No hay datos diarios: corre el notebook de producción con la casilla «Producción diaria».";
  else nota=(st.vista==="A"?"Pasa el cursor por una barra para ver el detalle; haz clic para ver sus meses."
             :"Pasa el cursor para ver el detalle de cada mes"+(DIA?"; los meses con reporte diario se abren con un clic.":"."))+ult+
            " Fuente: PERUPETRO, producción fiscalizada (medida en el punto de fiscalización); un mes en 0 puede ser paralización.";
  $("pp-nota").textContent=nota;
  $("pp-volver").style.display=st.pila.length?"":"none";
  if(st.pila.length)$("pp-volver").textContent="↩ "+(st.pila[st.pila.length-1].vista==="A"?"Serie anual":"Serie mensual");
  const ley=$("pp-leyenda");ley.textContent="";
  if(ss.length>1)ss.forEach((x,i)=>{const s=document.createElement("span"),k=document.createElement("i");k.style.background=COL[i];s.appendChild(k);s.appendChild(document.createTextNode(x.n));ley.appendChild(s);});
  resumen(ss,ds);grafico(ss,ds);tabla(ss,ds);
  $("pp-tabla-c").style.display=st.tabla?"block":"none";$("pp-btab").textContent=st.tabla?"Ocultar tabla":"Tabla";
}
function guardar(){st.pila.push({vista:st.vista,desde:st.desde,hasta:st.hasta,dmes:st.dmes});}
window.ppMeses=function(a){guardar();st.vista="M";st.dmes=null;st.desde=st.hasta=a;dibujar();};
window.ppDias=function(a,m){guardar();st.vista="D";st.dmes={a:a,m:m};st.desde=st.hasta=a;dibujar();};
window.ppVolver=function(){const p=st.pila.pop();if(p){st.vista=p.vista;st.desde=p.desde;st.hasta=p.hasta;st.dmes=p.dmes;}dibujar();};
window.ppTabla=function(){st.tabla=!st.tabla;dibujar();};
window.ppMin=function(){const p=$("pp-panel"),m=p.classList.toggle("pp-mini");$("pp-min").textContent=m?"+":"–";
  $("pp-fondo").style.display=m?"none":"block";                 // minimizado: el mapa vuelve a quedar libre
  $("pp-min").title=m?"Expandir":"Minimizar";$("pp-min").setAttribute("aria-label",m?"Expandir panel":"Minimizar panel");if(!m)dibujar();};
window.ppCerrar=function(){$("pp-panel").style.display="none";$("pp-fondo").style.display="none";$("pp-abrir").style.display="";};
window.ppAbrir=function(id,anio){
  if(id&&S[id])st.a=id;
  for(const k in window){try{const o=window[k];if(o&&window.L&&o instanceof L.Map)o.closePopup();}catch(e){}}  // la ficha no tapa el panel
  $("pp-abrir").style.display="none";$("pp-fondo").style.display="block";const p=$("pp-panel");p.style.display="flex";
  p.classList.remove("pp-mini");$("pp-min").textContent="–";st.pila=[];st.dmes=null;st.vista="A";st.desde=A0;st.hasta=A1;
  dibujar();if(anio)ppMeses(anio);
};
window.ppCSV=function(){const ss=series(),ds=ss.map(datos);
  let s="﻿"+[st.vista==="A"?"AÑO":st.vista==="M"?"MES":"FECHA"].concat(ss.map(x=>x.n+" ("+uni()+")")).join(";")+"\n";
  ds[0].forEach((r,i)=>{s+=[r.lab].concat(ds.map(d=>(Math.round(d[i].v*10)/10).toString().replace(".",","))).join(";")+"\n";});
  const a=document.createElement("a");a.href=URL.createObjectURL(new Blob([s],{type:"text/csv"}));
  a.download="produccion_"+ss.map(x=>x.n).join("_vs_").replace(/[^\w-]+/g,"_")+".csv";a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000);};
window.ppLey=function(){const c=$("pp-ley-c"),b=$("pp-ley-btn"),ab=c.style.display==="none";c.style.display=ab?"block":"none";
  b.textContent=ab?"–":"+";b.title=ab?"Minimizar leyenda":"Mostrar leyenda";b.setAttribute("aria-expanded",ab);};
function init(){opciones($("pp-a"),false);opciones($("pp-b"),true);anios();
  if(DIA){const o=document.createElement("option");o.value="D";o.textContent="Diaria";$("pp-vista").appendChild(o);}
  ["a","b","prod","vista","desde","hasta","u"].forEach(k=>$("pp-"+k).addEventListener("change",e=>{
    st[k]=(k==="desde"||k==="hasta")?+e.target.value:e.target.value;
    if(k==="vista"){st.pila=[];st.dmes=null;
      if(st.vista==="D"&&DIA&&(st.hasta<DA0||st.desde>DA1)){st.desde=DA0;st.hasta=DA1;}}   // la serie diaria es corta
    if(k==="desde"||k==="hasta")st.dmes=null;
    dibujar();}));
  let tm;window.addEventListener("resize",()=>{clearTimeout(tm);tm=setTimeout(()=>{if($("pp-panel").style.display==="flex")dibujar();},150);});
  document.addEventListener("keydown",ev=>{if(ev.key==="Escape"&&$("pp-panel").style.display==="flex")ppCerrar();});
  // la ficha (popup) de Leaflet: evitar que el clic en el botón se propague al mapa
  document.addEventListener("click",ev=>{const b=ev.target.closest&&ev.target.closest("[data-pp]");if(b){ev.preventDefault();ppAbrir(b.dataset.pp);}});
}
if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",init);else init();
})();
</script>
'''
ACT_HTML = r'''<style>
#pp-act-fondo{position:fixed;inset:0;background:rgba(0,0,0,.35);z-index:10000;display:none}
#pp-act{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);z-index:10001;display:none;background:#fff;border-radius:10px;
  box-shadow:0 4px 24px rgba(0,0,0,.35);width:min(760px,calc(100vw - 24px));max-height:calc(100vh - 24px);overflow:auto;
  font:12px/1.4 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:#1a1a1a}
#pp-act .cab{display:flex;align-items:flex-start;justify-content:space-between;gap:8px;padding:12px 14px 6px;border-bottom:1px solid #eee}
#pp-act .cab b{font-size:15px}
#pp-act .cab .sub{color:#666;font-size:12px}
#pp-act .x{border:0;background:#f3f2ee;border-radius:6px;width:28px;height:28px;cursor:pointer;font:600 16px sans-serif}
#pp-act .tabs{display:flex;gap:4px;padding:8px 14px 0;flex-wrap:wrap}
#pp-act .tabs button{border:1px solid #d6d4cc;background:#f7f6f2;border-radius:6px 6px 0 0;padding:5px 10px;cursor:pointer;font:600 12px sans-serif;color:#333}
#pp-act .tabs button[aria-selected=true]{background:#fff;border-bottom-color:#fff;color:#000}
#pp-act .tabs button:disabled{opacity:.4;cursor:default}
#pp-act .cuerpo{padding:8px 14px 14px;border-top:1px solid #d6d4cc;margin-top:-1px}
#pp-act .res{color:#333;margin:4px 0 6px}
#pp-act .ley{display:flex;gap:12px;flex-wrap:wrap;color:#444;margin:2px 0 4px}
#pp-act .ley i{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:4px;vertical-align:middle}
#pp-act svg{display:block;width:100%;height:auto;overflow:visible}
#pp-act svg .b{cursor:pointer}
#pp-act svg .sel{stroke:#000;stroke-width:1.5}
#pp-act .tip{position:fixed;pointer-events:none;background:#222;color:#fff;border-radius:5px;padding:5px 8px;font-size:11px;
  display:none;z-index:10002;white-space:nowrap}
#pp-act table{border-collapse:collapse;width:100%;margin-top:8px}
#pp-act th,#pp-act td{text-align:right;padding:3px 6px;border-bottom:1px solid #eee}
#pp-act th:first-child,#pp-act td:first-child{text-align:left}
#pp-act th{color:#555;font-weight:600;background:#faf9f6}
#pp-act tfoot td{font-weight:700}
#pp-act .nota{color:#777;margin-top:6px}
</style>
<div id="pp-act-fondo"></div>
<div id="pp-act" role="dialog" aria-modal="true" aria-labelledby="pp-act-tit">
  <div class="cab"><div><b id="pp-act-tit"></b><div class="sub" id="pp-act-sub"></div></div>
    <button type="button" class="x" title="Cerrar (Esc)" aria-label="Cerrar">×</button></div>
  <div class="tabs" role="tablist"></div>
  <div class="cuerpo"></div>
  <div class="tip"></div>
</div>
<script>
(function(){
const A=__ACT__;
const COL_P={d:"#1565c0",e:"#00897b"}, NOM_P={d:"Desarrollo",e:"Exploratorios y confirmatorios"};
const COL_R={"PETRÓLEO":"#6d4c41","LGN":"#ff8f00","GAS NATURAL":"#546e7a","CONDENSADO":"#bcaaa4"};
const COL_C="#7b1fa2";
const TABS=[["p","Pozos perforados"],["r","Regalías"],["c","Canon y sobrecanon"]];
const caja=document.getElementById("pp-act"),fondo=document.getElementById("pp-act-fondo"),tip=caja.querySelector(".tip");
let st=null;
function num(v,d){const s=Math.abs(v).toFixed(d||0).split(".");return (v<0?"-":"")+s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
function rango(k){const r=A.rango[k];const o=[];for(let a=r[0];a<=r[1];a++)o.push(a);return o;}
function titulo(s){return s.charAt(0)+s.slice(1).toLowerCase();}
// series apiladas por año: [{k, nombre, color, v:{año:valor}}]
function series(t){
  if(t==="p"){const d=A.p[st.c]||{};return ["d","e"].map(k=>{const v={};for(const l in d)for(const a in (d[l][k]||{}))v[a]=(v[a]||0)+d[l][k][a];
    return {k,nombre:NOM_P[k],color:COL_P[k],v};});}
  if(t==="r"){const d=A.r[st.c]||{};const hs=Object.keys(COL_R);return hs.map(h=>{const v={};for(const l in d)for(const a in (d[l][h]||{}))v[a]=(v[a]||0)+d[l][h][a];
    return {k:h,nombre:titulo(h),color:COL_R[h],v};}).filter(s=>Object.keys(s.v).length);}
  return [{k:"c",nombre:"Canon y sobrecanon",color:COL_C,v:A.c[st.d]||{}}];
}
const UNI={p:["pozos",0],r:["MM US$",2],c:["MM S/",1]};
function grafico(t){
  const ss=series(t),anios=rango(t),[u,dec]=UNI[t];
  const W=720,H=230,ml=44,mr=8,mt=10,mb=24,w=(W-ml-mr)/anios.length;
  const tot=a=>ss.reduce((x,s)=>x+(s.v[a]||0),0);let vmax=Math.max(...anios.map(tot))||1;
  const paso=Math.pow(10,Math.floor(Math.log10(vmax)));const nice=[1,2,2.5,5,10].map(f=>f*paso).find(x=>vmax/x<=5)||paso*10;
  vmax=Math.ceil(vmax/nice)*nice;
  const y=v=>mt+(H-mt-mb)*(1-v/vmax);let g="";
  for(let v=0;v<=vmax+1e-9;v+=nice)g+=`<line x1="${ml}" x2="${W-mr}" y1="${y(v)}" y2="${y(v)}" stroke="#e8e6e0"/><text x="${ml-6}" y="${y(v)+3}" font-size="10" fill="#666" text-anchor="end">${num(v,v<10&&dec?1:0)}</text>`;
  anios.forEach((a,i)=>{let base=0;const x=ml+i*w;
    ss.forEach(s=>{const v=s.v[a]||0;if(v<=0)return;const y1=y(base+v),y0=y(base);
      g+=`<rect class="b${a===st.anio?" sel":""}" data-a="${a}" x="${(x+1).toFixed(1)}" y="${y1.toFixed(1)}" width="${Math.max(w-2,1).toFixed(1)}" height="${Math.max(y0-y1,0.5).toFixed(1)}" fill="${s.color}" rx="1"/>`;base+=v;});
    g+=`<rect class="b" data-a="${a}" x="${x}" y="${mt}" width="${w}" height="${H-mt-mb}" fill="transparent"/>`;
    if(a%5===0||i===0||i===anios.length-1)g+=`<text x="${x+w/2}" y="${H-8}" font-size="10" fill="#666" text-anchor="middle">${a}</text>`;});
  const ley=ss.length>1?`<div class="ley">${ss.map(s=>`<span><i style="background:${s.color}"></i>${s.nombre}</span>`).join("")}</div>`:"";
  return {svg:`${ley}<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Gráfico anual">${g}</svg>`,ss,tot,u,dec};
}
function detalle(t,a){
  if(t==="p"){const d=A.p[st.c]||{};const ls=Object.keys(d).filter(l=>(d[l].d||{})[a]||(d[l].e||{})[a]);
    if(!ls.length)return `<div class="nota">Sin pozos perforados en ${a}.</div>`;let td=0,te=0;
    const f=ls.map(l=>{const x=(d[l].d||{})[a]||0,e=(d[l].e||{})[a]||0;td+=x;te+=e;return `<tr><td>Lote ${l}</td><td>${num(x)}</td><td>${num(e)}</td><td>${num(x+e)}</td></tr>`;}).join("");
    return `<table><thead><tr><th>${a}</th><th>Desarrollo</th><th>Exploratorios y confirmatorios</th><th>Total</th></tr></thead><tbody>${f}</tbody>${ls.length>1?`<tfoot><tr><td>Total</td><td>${num(td)}</td><td>${num(te)}</td><td>${num(td+te)}</td></tr></tfoot>`:""}</table>`;}
  if(t==="r"){const d=A.r[st.c]||{};const hs=Object.keys(COL_R);const ls=Object.keys(d).filter(l=>hs.some(h=>(d[l][h]||{})[a]));
    if(!ls.length)return `<div class="nota">Sin regalías registradas en ${a}.</div>`;const usadas=hs.filter(h=>ls.some(l=>(d[l][h]||{})[a]));const tt={};
    const f=ls.map(l=>{let s=0;const c=usadas.map(h=>{const v=(d[l][h]||{})[a]||0;s+=v;tt[h]=(tt[h]||0)+v;return `<td>${v?num(v,2):"–"}</td>`;}).join("");return `<tr><td>Lote ${l}</td>${c}<td>${num(s,2)}</td></tr>`;}).join("");
    const T=usadas.reduce((x,h)=>x+tt[h],0);
    return `<table><thead><tr><th>${a} (MM US$)</th>${usadas.map(h=>`<th>${titulo(h)}</th>`).join("")}<th>Total</th></tr></thead><tbody>${f}</tbody>${ls.length>1||usadas.length>1?`<tfoot><tr><td>Total</td>${usadas.map(h=>`<td>${num(tt[h],2)}</td>`).join("")}<td>${num(T,2)}</td></tr></tfoot>`:""}</table>`;}
  const c=A.c[st.d]||{};const v=c[a];const prev=c[a-1];
  return `<table><thead><tr><th>${a}</th><th>Millones de S/</th><th>Variación vs ${a-1}</th></tr></thead><tbody><tr><td>${titulo(st.d)}</td><td>${v!=null?num(v,2):"–"}</td><td>${v!=null&&prev?num((v/prev-1)*100,1)+" %":"–"}</td></tr></tbody></table>
    <div class="nota">El canon y sobrecanon es la parte del valor de la producción que el Estado transfiere al departamento (gobierno regional, municipios y universidades). Es del departamento entero, no solo de este lote.</div>`;
}
function resumen(t,G){const an=rango(t);const tot=an.map(G.tot);const s=tot.reduce((x,v)=>x+v,0);
  const conDato=an.filter((a,i)=>tot[i]>0);if(!conDato.length)return "Sin datos.";
  const imax=tot.indexOf(Math.max(...tot));
  return `Total ${an[0]}–${an[an.length-1]}: <b>${num(s,t==="p"?0:1)} ${G.u}</b> · máximo: <b>${num(tot[imax],t==="p"?0:1)}</b> en ${an[imax]} · último año con dato: ${conDato[conDato.length-1]}`;}
function tieneDatos(t){return t==="p"?!!A.p[st.c]:t==="r"?!!A.r[st.c]:!!A.c[st.d];}
function pintar(){
  const t=st.t,tabs=caja.querySelector(".tabs");
  tabs.innerHTML=TABS.map(([k,n])=>`<button type="button" role="tab" data-t="${k}" aria-selected="${k===t}" ${tieneDatos(k)?"":"disabled"}>${n}${k==="c"&&st.d?" · "+titulo(st.d):""}</button>`).join("");
  tabs.querySelectorAll("button").forEach(b=>b.onclick=()=>{st.t=b.dataset.t;st.anio=null;pintar();});
  const G=grafico(t);const an=rango(t);
  if(st.anio==null){const conDato=an.filter(a=>G.tot(a)>0);st.anio=conDato.length?conDato[conDato.length-1]:an[an.length-1];}
  const G2=grafico(t);
  caja.querySelector(".cuerpo").innerHTML=`<div class="res">${resumen(t,G2)}</div>${G2.svg}<div class="nota">Pasa el cursor por una barra para ver el valor; haz clic para ver el detalle de ese año.</div><div class="det">${detalle(t,st.anio)}</div>`;
  const svg=caja.querySelector("svg");
  svg.addEventListener("mousemove",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r){tip.style.display="none";return;}
    const a=+r.dataset.a;const partes=G2.ss.filter(s=>s.v[a]).map(s=>`${G2.ss.length>1?s.nombre+": ":""}${num(s.v[a],G2.dec)}`);
    tip.innerHTML=`<b>${a}</b>${partes.length?"<br>"+partes.join("<br>"):"<br>sin dato"}${G2.ss.length>1&&partes.length>1?`<br>Total: ${num(G2.tot(a),G2.dec)}`:""} ${G2.u}`;
    tip.style.display="block";tip.style.left=Math.min(ev.clientX+12,innerWidth-180)+"px";tip.style.top=(ev.clientY+12)+"px";});
  svg.addEventListener("mouseleave",()=>tip.style.display="none");
  svg.addEventListener("click",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r)return;st.anio=+r.dataset.a;pintar();});
}
function abrir(o){st={c:o.c,l:o.l,d:o.d,t:o.t,anio:null};if(!tieneDatos(st.t))st.t=TABS.map(x=>x[0]).find(tieneDatos)||"p";
  document.getElementById("pp-act-tit").textContent="Lote "+o.l;
  const lotes=Object.keys(Object.assign({},A.p[o.c]||{},A.r[o.c]||{}));
  document.getElementById("pp-act-sub").textContent="Campo "+o.c+(lotes.length>1?" (lotes "+lotes.join(", ")+")":"")+(o.d?" · "+titulo(o.d):"");
  caja.style.display="block";fondo.style.display="block";pintar();caja.querySelector(".x").focus();}
function cerrar(){caja.style.display="none";fondo.style.display="none";tip.style.display="none";}
caja.querySelector(".x").onclick=cerrar;fondo.onclick=cerrar;
document.addEventListener("keydown",e=>{if(e.key==="Escape"&&caja.style.display==="block")cerrar();});
document.addEventListener("click",ev=>{const b=ev.target.closest&&ev.target.closest("[data-act]");if(!b||caja.contains(b))return;
  ev.preventDefault();try{abrir(JSON.parse(b.dataset.act));}catch(e){}});
window.ppActAbrir=abrir;
})();
</script>
'''
def series_panel(prod, geo, gas=None):
    if prod is None or not len(prod):
        return None
    p_ = prod.copy()
    p_["BARRILES"] = pd.to_numeric(p_["BARRILES"], errors="coerce").fillna(0)
    if gas is not None and len(gas):                    # el gas entra como un producto más (MPC en la columna BARRILES)
        g_ = gas.rename(columns={"MPC": "BARRILES"}).copy()
        g_["BARRILES"] = pd.to_numeric(g_["BARRILES"], errors="coerce").fillna(0)
        g_["PRODUCTO"] = "GAS NATURAL"
        p_ = pd.concat([p_, g_[[c for c in g_.columns if c in p_.columns]]], ignore_index=True)
    p_["_T"] = (p_["AÑO"].astype(int) - ANIO_INI) * 12 + p_["MES"].astype(int) - 1
    n = int(p_.loc[p_["BARRILES"] > 0, "_T"].max()) + 1                     # hasta el último mes con datos
    p_ = p_[(p_["_T"] >= 0) & (p_["_T"] < n)]
    _pr = p_["PRODUCTO"].astype(str).str.upper()
    p_["_PET"] = _pr.map(lambda x: 0 if x.startswith("PETR") else (2 if x.startswith("GAS") else 1))   # 0 petróleo, 1 LGN/cond., 2 gas
    p_["LOTE"] = p_["LOTE"].astype(str)
    base = p_["LOTE_HISTORICO"] if "LOTE_HISTORICO" in p_ else p_["LOTE"]
    p_["_CAMPO"] = (p_["CAMPO"] if "CAMPO" in p_ else base).fillna(base).astype(str)
    dep_geo = dict(zip(geo["LOTE"].astype(str), geo.get("DEPARTAMENTO", pd.Series("", index=geo.index)).fillna("")))
    p_["_DEP"] = p_["DEPARTAMENTO"].fillna("").astype(str) if "DEPARTAMENTO" in p_ else ""
    p_.loc[p_["_DEP"].str.strip() == "", "_DEP"] = p_["LOTE"].map(dep_geo).fillna("")
    p_["_PERU"] = "PERÚ"
    lotes_por_campo = p_.groupby("_CAMPO")["LOTE"].nunique()
    salida = []
    for grupo, col, pref, con_ops in [("Total", "_PERU", "T", False), ("Departamentos", "_DEP", "D", False),
                                       ("Campos (lotes sucesivos sumados)", "_CAMPO", "C", True), ("Lotes", "LOTE", "L", True)]:
        q = p_[p_[col].astype(str).str.strip() != ""]
        if pref == "L":                                   # un lote solo aparece aparte si su campo tiene varios lotes
            q = q[q["_CAMPO"].map(lotes_por_campo).fillna(1) > 1]
        if q.empty:
            continue
        t = q.groupby([col, "_PET", "_T"])["BARRILES"].sum().unstack(fill_value=0).reindex(columns=range(n), fill_value=0)
        ops = {}
        if con_ops and "OPERADOR" in q:
            o_ = q[(q["BARRILES"] > 0) & (q["OPERADOR"].fillna("").astype(str).str.strip() != "")]
            for (k, a), s_ in o_.groupby([col, "AÑO"])["OPERADOR"]:
                ops.setdefault(k, {})[int(a)] = " / ".join(sorted(set(s_.astype(str))))
        orden = q[q["_PET"] < 2].groupby(col)["BARRILES"].sum()          # de mayor a menor producción de líquidos
        for k in sorted(q[col].unique(), key=lambda x: (-float(orden.get(x, 0)), str(x))):
            serie = lambda pet: [int(round(float(v))) for v in t.loc[(k, pet)]] if (k, pet) in t.index else [0] * n
            nombre = {"T": "Perú (total)", "D": str(k).title(), "C": f"Campo {k}", "L": f"Lote {k}"}[pref]
            gs = serie(2)
            salida.append({"id": f"{pref}:{k}", "g": grupo, "n": nombre, "p": serie(0), "l": serie(1), **({"gs": gs} if any(gs) else {}),
                           **({"o": ops.get(k, {})} if con_ops else {})})
    if not salida:
        return None
    defecto = next((x["id"] for x in salida if x["id"] == "D:LORETO"), salida[0]["id"])
    dia = series_diarias(diaria, salida, dict(zip(p_["LOTE"], p_["_CAMPO"])), dep_geo) if diaria is not None else None
    return {"ini": ANIO_INI, "n": n, "s": salida, "def": defecto, "dia": dia,
            "grupos": ["Total", "Departamentos", "Campos (lotes sucesivos sumados)", "Lotes"]}

def series_diarias(diaria, salida, campo_de, dep_de):
    """Agrega a cada serie del panel su producción DIARIA (dd.p petróleo y condensado, dd.l LGN en bbl; dd.g gas en Mcf)."""
    d = diaria[diaria["UNIDAD"].isin(["BBL", "MCF"])].copy()
    if d.empty:
        return None
    d["VALOR"] = pd.to_numeric(d["VALOR"], errors="coerce").fillna(0)
    d["_F"] = pd.to_datetime(d["FECHA"], format="%d/%m/%Y")
    ini = d["_F"].min(); nd = int((d["_F"].max() - ini).days) + 1
    d["_i"] = (d["_F"] - ini).dt.days
    d["_K"] = ["g" if u == "MCF" else ("l" if str(p).upper().startswith("LGN") else "p") for u, p in zip(d["UNIDAD"], d["PRODUCTO"])]
    d["LOTE"] = d["LOTE"].astype(str)
    d["_CAMPO"] = d["LOTE"].map(lambda l: campo_de.get(l, CAMPO_DE.get(l, l)) if "CAMPO_DE" in globals() else campo_de.get(l, l))
    dep = d["DEPARTAMENTO"].fillna("").astype(str) if "DEPARTAMENTO" in d else pd.Series("", index=d.index)
    d["_DEP"] = [x if x.strip() else dep_de.get(l, "") for x, l in zip(dep, d["LOTE"])]
    d["_PERU"] = "PERÚ"
    por_id = {x["id"]: x for x in salida}
    for col, pref in [("_PERU", "T"), ("_DEP", "D"), ("_CAMPO", "C"), ("LOTE", "L")]:
        t = d[d[col].astype(str).str.strip() != ""].groupby([col, "_K", "_i"])["VALOR"].sum()
        for (k, kk), s_ in t.groupby(level=[0, 1]):
            x = por_id.get(f"{pref}:{k}")
            if x is None:
                continue
            arr = [0] * nd
            for i, v in zip(s_.index.get_level_values(2), s_.values):
                arr[int(i)] = int(round(float(v)))
            x.setdefault("dd", {})[kk] = arr
    return {"ini": [int(ini.year), int(ini.month), int(ini.day)], "n": nd}

diaria = globals().get("diaria")
DATOS_PANEL = series_panel(prod, geo, globals().get("gas")) if prod is not None else None
IDS_PANEL = {x["id"] for x in DATOS_PANEL["s"]} if DATOS_PANEL else set()

PRODUCTOS = [("p", "Petróleo", "bpd"), ("l", "LGN y condensado", "bpd"), ("g", "Gas natural", "MMPCD")]

def estados_producto(datos):
    """{campo: {"p"|"l"|"g": {"e": estado, "u": última producción mm/aaaa, "r": promedio diario de los últimos 12 meses}}}.
    En producción = produjo en los últimos 12 meses con datos; paralizado = produjo antes; sin producción = nunca.
    Los meses posteriores a la última estadística mensual se toman de los reportes diarios."""
    import calendar, datetime as _dt
    if not datos:
        return {}
    ini, n, dia = datos["ini"], datos["n"], datos.get("dia")
    n_tot, d0 = n, None
    if dia:
        d0 = _dt.date(*dia["ini"]); ult = d0 + _dt.timedelta(days=dia["n"] - 1)
        n_tot = max(n, (ult.year - ini) * 12 + ult.month)
    dias_mes = [calendar.monthrange(ini + t // 12, t % 12 + 1)[1] for t in range(n_tot)]
    out = {}
    for x in datos["s"]:
        if not x["id"].startswith("C:"):
            continue
        res = {}
        for k, clave in (("p", "p"), ("l", "l"), ("g", "gs")):
            mens = list(x.get(clave) or [0] * n) + [0] * (n_tot - n)
            dias = list(dias_mes)
            if dia and n_tot > n:
                cubiertos = [0] * (n_tot - n)
                for i, v in enumerate((x.get("dd") or {}).get(k) or [0] * dia["n"]):
                    f = d0 + _dt.timedelta(days=i); t = (f.year - ini) * 12 + f.month - 1
                    if t >= n:
                        mens[t] += v; cubiertos[t - n] += 1
                for j, c in enumerate(cubiertos):
                    dias[n + j] = c or dias[n + j]
            pos = [t for t, v in enumerate(mens) if v > 0]
            if not pos:
                res[k] = {"e": "SIN PRODUCCIÓN"}
                continue
            L, fin = pos[-1], n_tot - 1
            ult12 = range(max(0, fin - 11), fin + 1)
            r = sum(mens[t] for t in ult12) / sum(dias[t] for t in ult12)
            res[k] = {"e": "EN PRODUCCIÓN" if L > fin - 12 else "PARALIZADO", "u": f"{L % 12 + 1:02d}/{ini + L // 12}",
                      "r": round(r / 1000, 2) if k == "g" else round(r, 0)}
        out[x["id"][2:]] = res
    return out

EST_PRODUCTO = estados_producto(DATOS_PANEL)
if EST_PRODUCTO:                                       # tabla para usar fuera del mapa
    _filas = []
    for _c, _e in sorted(EST_PRODUCTO.items()):
        _f = {"FECHA_EXTRACCION": date.today().strftime("%d/%m/%Y"), "CAMPO": _c}
        for _k, _n in (("p", "PETROLEO"), ("l", "LGN_CONDENSADO"), ("g", "GAS")):
            _d = _e.get(_k, {})
            _f.update({f"ESTADO_{_n}": _d.get("e", "SIN PRODUCCIÓN"), f"ULTIMA_PRODUCCION_{_n}": _d.get("u", ""),
                       f"ACTUAL_{_n}" + ("_MMPCD" if _k == "g" else "_BPD"): _d.get("r")})
        _filas.append(_f)
    _p = os.path.join(ruta, "PP_LOTES_ESTADO_PRODUCTO.csv")
    pd.DataFrame(_filas).to_csv(_p, index=False, encoding="utf-8-sig")
    subir(_p, DRIVE_FOLDER_ID, datos_drive)

def estado_producto_html(campo):
    e = EST_PRODUCTO.get(str(campo))
    if not e:
        return ""
    filas = ""
    for k, nombre, uni in PRODUCTOS:
        d = e.get(k, {"e": "SIN PRODUCCIÓN"})
        if d["e"] == "EN PRODUCCIÓN":
            txt = f"En producción · {_num(d['r'], 2 if k == 'g' else 0)} {uni}"
        elif d["e"] == "PARALIZADO":
            txt = f"Paralizado · última producción {d['u']}"
        else:
            txt = "Sin producción"
        col = COLOR_ESTADO.get("SIN PRODUCCIÓN REGISTRADA" if d["e"] == "SIN PRODUCCIÓN" else d["e"], "#9e9e9e")
        filas += (f"<tr><td style='color:#666;padding-right:8px'>{nombre}</td><td><span style='display:inline-block;width:9px;"
                  f"height:9px;border-radius:2px;background:{col};margin-right:5px'></span>{_h.escape(txt)}</td></tr>")
    return (f"<table style='margin-top:6px;border-top:1px solid #eee;padding-top:4px'><tr><td colspan=2 style='color:#333'>"
            f"<b>Estado por producto</b> <span style='color:#888'>(promedio de los últimos 12 meses)</span></td></tr>{filas}</table>")

# ── Pozos perforados, regalías y canon (del notebook de producción, sección 6.1) ──
def _leer_extra(nombre):
    if nombre not in datos_drive:
        return None
    p = os.path.join(ruta, nombre); bajar_de_drive(datos_drive[nombre], p)
    return pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_HISTORICO": str})

def _campo_de_lote(df):
    base = df["LOTE_HISTORICO"].astype(str) if "LOTE_HISTORICO" in df else df["LOTE"].astype(str)
    return [CAMPO_DE.get(l, b) for l, b in zip(df["LOTE"].astype(str), base)]

def _dep(t):
    return "".join(c for c in unicodedata.normalize("NFKD", str(t)) if not unicodedata.combining(c)).upper().strip()

# ACT: datos del panel «Pozos, regalías y canon» (un solo bloque JSON en el HTML)
#   p[campo][lote] = {"d": {año: pozos}, "e": {...}}; r[campo][lote][hidrocarburo] = {año: MM US$}; c[departamento] = {año: MM S/}
ACT = {"p": {}, "r": {}, "c": {}, "rango": {}}
POZOS_CAMPO, REGALIAS_CAMPO = {}, {}
_pz = _leer_extra("PP_pozos_perforados_lote_anio.csv")
if _pz is not None and len(_pz):
    _pz["_C"] = _campo_de_lote(_pz)
    _pz["_T"] = _pz["TIPO"].map(lambda t: "d" if str(t).startswith("DESARROLLO") else "e")   # confirmatorios con exploratorios
    for (c, l, t), g in _pz.groupby(["_C", "LOTE", "_T"]):
        v = g.groupby("AÑO")["POZOS"].sum()
        v = {int(a): int(n) for a, n in v.items() if n}
        if v:
            ACT["p"].setdefault(c, {}).setdefault(l, {})[t] = v
    for c, d in ACT["p"].items():
        POZOS_CAMPO[c] = {t: {} for t in "de"}
        for l in d.values():
            for t, v in l.items():
                for a, n in v.items():
                    POZOS_CAMPO[c][t][a] = POZOS_CAMPO[c][t].get(a, 0) + n
    ACT["rango"]["p"] = [int(_pz["AÑO"].min()), int(_pz["AÑO"].max())]
_rg = _leer_extra("PP_regalias_lote_anio.csv")
if _rg is not None and len(_rg):
    _rg["_C"] = _campo_de_lote(_rg)
    for (c, l, h), g in _rg.groupby(["_C", "LOTE", "HIDROCARBURO"]):
        v = (g.groupby("AÑO")["REGALIA_US$"].sum() / 1e6).round(3)
        ACT["r"].setdefault(c, {}).setdefault(l, {})[h] = {int(a): float(x) for a, x in v.items() if x}
    for c, g in _rg.groupby("_C"):
        REGALIAS_CAMPO[c] = (g.groupby("AÑO")["REGALIA_US$"].sum() / 1e6).round(2).to_dict()
    ACT["rango"]["r"] = [int(_rg["AÑO"].min()), int(_rg["AÑO"].max())]
_cn = _leer_extra("PP_canon_departamento_anio.csv")
if _cn is not None and len(_cn):
    for d, g in _cn.groupby(_cn["DEPARTAMENTO"].map(_dep)):
        ACT["c"][d] = {int(a): round(float(x), 2) for a, x in zip(g["AÑO"], g["CANON_MM_SOLES"])}
    ACT["rango"]["c"] = [int(_cn["AÑO"].min()), int(_cn["AÑO"].max())]
print(f"🛠️ Pozos: {len(POZOS_CAMPO)} campos · 💵 Regalías: {len(REGALIAS_CAMPO)} campos · 🏛️ Canon: {len(ACT['c'])} departamentos")

def barras_svg(series, a0, a1, unidad, dec=0, ancho=220, alto=46):
    """Barras anuales (apiladas si hay varias series): series = [(dict año→valor, color, nombre)]."""
    anios = list(range(a0, a1 + 1)); w = ancho / len(anios)
    tot = {a: sum(s.get(a, 0) for s, _, _ in series) for a in anios}; vmax = max(tot.values()) or 1
    h = alto - 16; out = ""
    for i, a in enumerate(anios):
        y = alto - 12
        tip = f"{a}: " + " · ".join(f"{n} {_num(s.get(a, 0), dec)}" if n else _num(s.get(a, 0), dec) for s, _, n in series) + f" {unidad}"
        for s, col, _ in series:
            v = s.get(a, 0)
            if v <= 0:
                continue
            bh = v / vmax * h; y -= bh
            out += f'<rect x="{i * w:.1f}" y="{y:.1f}" width="{max(w - 1, 1):.1f}" height="{bh:.1f}" fill="{col}"><title>{_h.escape(tip)}</title></rect>'
    return (f'<svg width="{ancho}" height="{alto}" xmlns="http://www.w3.org/2000/svg">{out}'
            f'<text x="0" y="{alto - 1}" font-size="9" fill="#555">{anios[0]}</text>'
            f'<text x="{ancho}" y="{alto - 1}" font-size="9" fill="#555" text-anchor="end">{anios[-1]}</text></svg>')

COLOR_DES, COLOR_EXP, COLOR_REG, COLOR_CAN = "#1565c0", "#00897b", "#6d4c41", "#7b1fa2"

def actividad_html(campo, lote, dep):
    """Pozos perforados, regalías del campo y canon del departamento, con mini gráficos; un clic abre el panel interactivo."""
    out, campo, dep = "", str(campo), _dep(dep or "")
    clic = lambda tab: _h.escape(json.dumps({"c": campo, "l": str(lote), "d": dep, "t": tab}, ensure_ascii=False), quote=True)
    caja = ("<div data-act='{}' title='Clic para ver el detalle por año y por lote' style='cursor:pointer;margin-top:8px;"
            "border-top:1px solid #eee;padding-top:4px'>")
    pz = POZOS_CAMPO.get(campo)
    if pz and (sum(pz["d"].values()) + sum(pz["e"].values())) > 0:
        d, e = pz["d"], pz["e"]; a0, a1 = ACT["rango"]["p"]
        ult = max(a for a in set(d) | set(e) if d.get(a, 0) + e.get(a, 0) > 0)
        out += (caja.format(clic("p")) + f"<b>Pozos perforados</b> <span style='color:#888'>({a0}–{a1})</span><br>"
                f"<span style='display:inline-block;width:9px;height:9px;background:{COLOR_DES};margin-right:3px'></span>Desarrollo: "
                f"<b>{_num(sum(d.values()), 0)}</b> · <span style='display:inline-block;width:9px;height:9px;background:{COLOR_EXP};"
                f"margin-right:3px'></span>Exploratorios y confirmatorios: <b>{_num(sum(e.values()), 0)}</b> · último: {ult}<br>"
                + barras_svg([(d, COLOR_DES, "desarrollo"), (e, COLOR_EXP, "exploratorios")], a0, a1, "pozos") + "</div>")
    rg = REGALIAS_CAMPO.get(campo)
    if rg and sum(rg.values()) > 0:
        a0, a1 = ACT["rango"]["r"]
        out += (caja.format(clic("r")) + f"<b>Regalías pagadas al Estado</b> <span style='color:#888'>(millones de US$, {a0}–{a1})"
                f"</span><br>Total: <b>{_num(sum(rg.values()), 1)}</b> · {a1}: <b>{_num(rg.get(a1, 0), 1)}</b><br>"
                + barras_svg([(rg, COLOR_REG, "")], a0, a1, "MM US$", dec=1) + "</div>")
    cn = ACT["c"].get(dep)
    if cn:
        a0, a1 = ACT["rango"]["c"]
        out += (caja.format(clic("c")) + f"<b>Canon y sobrecanon de {_h.escape(dep.title())}</b> <span style='color:#888'>"
                f"(millones de S/, {a0}–{a1}; del departamento, no solo de este lote)</span><br>{a1}: <b>{_num(cn.get(a1, 0), 1)}</b> · "
                f"máximo: <b>{_num(max(cn.values()), 1)}</b> ({max(cn, key=cn.get)})<br>"
                + barras_svg([(cn, COLOR_CAN, "")], a0, a1, "MM S/", dec=1) + "</div>")
    if out:
        out += (f"<button type='button' data-act='{clic('p' if pz else 'r' if rg else 'c')}' style='margin-top:4px;border:1px solid #c9c8c2;"
                "background:#f3f2ee;border-radius:5px;padding:3px 8px;cursor:pointer;font:600 12px sans-serif'>"
                "📊 Ver pozos, regalías y canon</button>")
    return out

def popup_html(r):
    est = r.get("ESTADO_PRODUCCION", "")
    filas = ""
    for c, e, u in ETIQUETAS:
        x = fmt(c, r.get(c), u) if c in r else None
        if c in ("PARALIZACION_MAX_MESES", "PARALIZACION_MAX_DESDE") and not r.get("PARALIZACION_MAX_MESES"):
            x = None                                    # sin paralizaciones: no se muestra
        if c == "GAS_ACUM_CAMPO_BCF" and not (pd.notna(r.get(c)) and float(r.get(c) or 0) > 0):
            x = None                                    # campos sin gas fiscalizado
        if x is not None:
            filas += f"<tr><td style='color:#666;padding-right:8px;vertical-align:top'>{e}</td><td>{x}</td></tr>"
    serie = ind_campo.get(r.get("CAMPO"), {}).get("_SERIE_ANUAL_MMBBL", {})
    graf = (f"<div style='margin-top:6px;color:#666'>Producción fiscalizada anual de líquidos (MM bbl)</div>{sparkline_svg(serie, ANIO_INI, ANIO_FIN)}"
            if serie else "")
    id_ = f"C:{r.get('CAMPO')}"
    if graf and id_ in IDS_PANEL:                      # la línea de tiempo abre el panel con selectores y gráficos
        graf = (f"<div data-pp='{_h.escape(id_, quote=True)}' style='cursor:pointer' title='Ver producción fiscalizada anual y mensual (líquidos y gas)'>{graf}</div>"
                f"<button type='button' data-pp='{_h.escape(id_, quote=True)}' style='margin-top:4px;border:1px solid #c9c8c2;"
                "background:#f3f2ee;border-radius:5px;padding:3px 8px;cursor:pointer;font:600 12px sans-serif'>"
                "📊 Ver producción anual y mensual</button>")
    return (f"<div style='font:12px sans-serif;min-width:260px'><b style='font-size:14px'>Lote {_h.escape(str(r['LOTE']))}</b>"
            f"{estado_producto_html(r.get('CAMPO'))}<table style='margin-top:6px'>{filas}</table>{graf}"
            f"{actividad_html(r.get('CAMPO') if pd.notna(r.get('CAMPO')) else r['LOTE'], r['LOTE'], r.get('DEPARTAMENTO'))}</div>")

# ── Capas de contexto ────────────────────────────────────────────────────────
if deps is not None:
    folium.GeoJson(simpl(deps, 0.005), name="Departamentos",
                   style_function=lambda f: {"color": "#666", "weight": 0.8, "fillOpacity": 0},
                   tooltip=folium.GeoJsonTooltip(["DEPARTAMENTO"], labels=False)).add_to(m)
if anp is not None:
    a_ = simpl(anp[[c for c in [anp_n] if c] + ["geometry"]])
    folium.GeoJson(a_, name="Áreas Naturales Protegidas", show=False,
                   style_function=lambda f: {"color": "#2e7d32", "weight": 0.5, "fillColor": "#66bb6a", "fillOpacity": 0.25},
                   tooltip=folium.GeoJsonTooltip([anp_n], labels=False) if anp_n else None).add_to(m)
CLASES_RIO = {"Afluente (1.000–5.000 m³/s)": 1, "Río troncal (5.000–20.000 m³/s)": 2.2,
              "Amazonas / tramo bajo (> 20.000 m³/s)": 3.5}
if rios_q is not None:
    r_ = rios_q[rios_q["DIS_AV_CMS"] >= 1000].copy()
    r_["CLASE"] = pd.cut(r_["DIS_AV_CMS"], [999, 5000, 20000, 1e9], labels=list(CLASES_RIO)).astype(str)
    r_ = simpl(r_[["CLASE", "geometry"]].dissolve("CLASE", as_index=False), 0.003)
    folium.GeoJson(r_, name="Ríos (HydroRIVERS)",
                   style_function=lambda f: {"color": "#1e88e5", "weight": CLASES_RIO.get(f["properties"]["CLASE"], 1), "opacity": 0.8},
                   tooltip=folium.GeoJsonTooltip(["CLASE"], labels=False)).add_to(m)
if ccnn is not None and ccnn_n:
    mc = MarkerCluster(name="Comunidades nativas", show=False)
    for _, r in ccnn.to_crs(4326).iterrows():
        if r.geometry is None or r.geometry.is_empty:
            continue
        pt = r.geometry.representative_point()
        folium.CircleMarker([pt.y, pt.x], radius=3, color="#8e24aa", fill=True, fill_opacity=0.8,
                            tooltip=_h.escape(str(r[ccnn_n]))).add_to(mc)
    mc.add_to(m)

if onp is not None:
    o_ = simpl(onp[["geometry"]].dissolve(), 0.002)
    o_["NOMBRE"] = "Oleoducto Norperuano"
    folium.GeoJson(o_, name="Oleoducto Norperuano",
                   style_function=lambda f: {"color": "#111", "weight": 3, "dashArray": "6 4"},
                   tooltip=folium.GeoJsonTooltip(["NOMBRE"], labels=False)).add_to(m)
# ── Lotes: una sola capa; su color y los círculos de producción actual dependen de los productos marcados ──
capa_l = geo[geo["GEOMETRIA_DE"] == geo["LOTE"]].copy() if "GEOMETRIA_DE" in geo else geo.copy()
capa_l = simpl(capa_l, 0.001)
fg_lotes = folium.FeatureGroup(name="Lotes", control=False)
CENTROS = {}
for _, r in capa_l.iterrows():
    campo = str(r.get("CAMPO") if pd.notna(r.get("CAMPO")) else r["LOTE"])
    gj = folium.GeoJson({"type": "Feature", "geometry": r.geometry.__geo_interface__,
                         "properties": {"pp_campo": campo, "pp_lote": str(r["LOTE"])}},
                        style_function=lambda f: {"color": "#9e9e9e", "weight": 1.2, "fillColor": "#9e9e9e", "fillOpacity": 0.45})
    folium.Popup(popup_html(r), max_width=380).add_to(gj)
    gj.add_to(fg_lotes)
    if campo not in CENTROS and pd.notna(r.get("LAT")):
        CENTROS[campo] = [round(float(r["LAT"]), 4), round(float(r["LON"]), 4)]
fg_lotes.add_to(m)
folium.Marker([IQUITOS.y, IQUITOS.x], tooltip="Iquitos", icon=folium.Icon(color="blue", icon="info-sign")).add_to(m)

caja = "display:inline-block;width:11px;height:11px;margin-right:5px;vertical-align:middle"
# Leyenda en la esquina inferior derecha: cerrada al abrir el mapa; el botón + / – la abre y la minimiza
leyenda = ("<div id='pp-ley' class='pp-caja'><div id='pp-ley-cab'><b>Leyenda</b>"
           "<button type='button' class='pp-btn' id='pp-ley-btn' title='Mostrar leyenda' aria-expanded='false' "
           "onclick='ppLey()'>+</button></div><div id='pp-ley-c' style='display:none;margin-top:4px'>"
           + "<span style='color:#555'>Estado según los <b>productos</b> marcados en el selector de capas:</span><br>"
           + "".join(f"<span style='{caja};background:{COLOR_ESTADO[e]};opacity:.7'></span>{t}<br>"
                     for e, t in [("EN PRODUCCIÓN", "En producción (alguno de los productos)"), ("PARALIZADO", "Paralizado"),
                                  ("SIN PRODUCCIÓN REGISTRADA", "Sin producción")])
           + f"<span style='{caja};border-radius:50%;background:#66bb6a'></span>Producción actual (círculo ∝ bep/d; "
             "gas: 6 mil pies³ = 1 bep)<br>"
           "<span style='display:inline-block;width:20px;border-top:3px dashed #111;margin-right:5px;vertical-align:middle'></span>"
           "Oleoducto Norperuano<br>"
           "<span style='display:inline-block;width:20px;border-top:3px solid #1e88e5;margin-right:5px;vertical-align:middle'></span>"
           "Ríos grandes<br><span style='color:#777'>En producción = produjo en los últimos 12 meses. "
           "Clic en un lote o círculo para ver su ficha (con el estado de cada producto); en la ficha, clic en la línea de tiempo para ver la producción anual, mensual y diaria."
           "</span></div></div>")
m.get_root().html.add_child(folium.Element(leyenda))
if DATOS_PANEL:
    _js = json.dumps(DATOS_PANEL, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    # folium.Element trata el texto como plantilla Jinja: {% raw %} evita que lea "{#" del CSS/JS como comentario
    m.get_root().html.add_child(folium.Element("{% raw %}" + PANEL_HTML.replace("__DATA__", _js) + "{% endraw %}"))
else:                                                  # sin producción: solo los estilos de la leyenda
    m.get_root().html.add_child(folium.Element("{% raw %}" + PANEL_HTML.split("</style>")[0] + "</style>"
                                               "<script>window.ppLey=function(){var c=document.getElementById('pp-ley-c'),"
                                               "b=document.getElementById('pp-ley-btn'),a=c.style.display==='none';"
                                               "c.style.display=a?'block':'none';b.textContent=a?'–':'+';};</script>{% endraw %}"))
# panel «Pozos, regalías y canon» (se abre desde la ficha del lote)
if ACT["p"] or ACT["r"] or ACT["c"]:
    _ja = json.dumps({**ACT, "p": ACT["p"], "r": ACT["r"]}, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    m.get_root().html.add_child(folium.Element("{% raw %}" + ACT_HTML.replace("__ACT__", _ja) + "{% endraw %}"))
folium.LayerControl(collapsed=False).add_to(m)
# selector de productos (dentro del control de capas): colorea los lotes y dibuja los círculos de producción actual
PRODUCTOS_HTML = r'''<style>
.pp-prod-sel{margin-top:4px;padding-top:4px;border-top:1px solid #ddd}
.pp-prod-sel b{display:block;margin:2px 0 3px}
.pp-prod-sel label{display:block;cursor:pointer}
/* sin el rectángulo de foco del navegador al hacer clic en un polígono */
path.leaflet-interactive:focus{outline:none}
</style>
<script>
(function(){
const EST=__EST__, CENTROS=__CENTROS__;
const COL={"EN PRODUCCIÓN":"#2e7d32","PARALIZADO":"#ef6c00","SIN PRODUCCIÓN":"#9e9e9e"};
const NOMBRE={p:"petróleo y condensado",l:"LGN",g:"gas natural"}, UNI={p:"bpd",l:"bpd",g:"MMPCD"};
const sel={p:true,l:true,g:true};
let mapa=null,lotes=[],circulos=null;
function num(v,d){const s=Math.abs(v).toFixed(d||0).split(".");return s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
// estado combinado de los productos marcados: en producción si alguno produce; paralizado si alguno produjo antes
function estado(campo){
  const e=EST[campo]||{};let hay=false,par=false;
  for(const k in sel){if(!sel[k])continue;const d=e[k];if(!d)continue;
    if(d.e==="EN PRODUCCIÓN")hay=true;else if(d.e==="PARALIZADO")par=true;}
  return hay?"EN PRODUCCIÓN":par?"PARALIZADO":"SIN PRODUCCIÓN";
}
function detalle(campo){
  const e=EST[campo]||{},partes=[];
  for(const k in sel){if(!sel[k])continue;const d=e[k];if(!d||d.e==="SIN PRODUCCIÓN")continue;
    partes.push(NOMBRE[k]+": "+(d.e==="EN PRODUCCIÓN"?num(d.r,k==="g"?2:0)+" "+UNI[k]:"paralizado desde "+d.u));}
  return partes.join(" · ");
}
function bep(campo){                                   // barriles equivalentes por día de los productos marcados
  const e=EST[campo]||{};let s=0;
  for(const k in sel){const d=e[k];if(sel[k]&&d&&d.e==="EN PRODUCCIÓN")s+=k==="g"?d.r*1000/6:d.r;}
  return s;
}
function aplicar(){
  // con la línea de tiempo animada activa (mapa con plugins), ella pinta los lotes
  if(window.ppTL&&window.ppTL.activo){window.ppTL.pintar();return;}
  const ninguno=!sel.p&&!sel.l&&!sel.g;
  lotes.forEach(o=>{
    const el=o.capa.getElement&&o.capa.getElement();
    if(ninguno){                                       // sin productos marcados: los lotes no se muestran
      o.capa.setStyle({opacity:0,fillOpacity:0});if(el)el.style.pointerEvents="none";o.capa.closeTooltip();return;}
    const est=estado(o.campo),c=COL[est];
    o.capa.setStyle({color:c,fillColor:c,opacity:1,fillOpacity:0.45,weight:1.2});if(el)el.style.pointerEvents="";
    const det=detalle(o.campo);
    o.capa.setTooltipContent("Lote "+o.lote+" · "+est.toLowerCase()+(det?"<br>"+det:""));
  });
  circulos.clearLayers();
  if(ninguno)return;
  Object.keys(CENTROS).forEach(campo=>{
    const v=bep(campo);if(!(v>0))return;
    const ll=CENTROS[campo],grupo=lotes.find(o=>o.campo===campo);
    const c=L.circleMarker(ll,{radius:4+Math.sqrt(v)/10,color:"#1b5e20",weight:1,fillColor:"#66bb6a",fillOpacity:0.75});
    c.bindTooltip("Campo "+campo+": "+num(v,0)+" bep/d<br>"+detalle(campo)+"<br><i>clic para la ficha</i>");
    if(grupo)c.on("click",()=>grupo.grupo.openPopup(L.latLng(ll)));      // el círculo abre la ficha del lote
    circulos.addLayer(c);
  });
}
function iniciar(){
  for(const k in window){try{if(window[k] instanceof L.Map){mapa=window[k];break;}}catch(e){}}
  if(!mapa)return;
  mapa.eachLayer(g=>{
    if(!(g instanceof L.GeoJSON))return;
    g.eachLayer(c=>{const p=c.feature&&c.feature.properties;
      if(p&&p.pp_campo!==undefined){c.bindTooltip("");lotes.push({campo:String(p.pp_campo),lote:p.pp_lote,capa:c,grupo:g});}});
  });
  circulos=L.featureGroup().addTo(mapa);
  // casillas de productos dentro del selector de capas
  const lista=document.querySelector(".leaflet-control-layers-list");
  if(lista){
    const caja=document.createElement("div");caja.className="pp-prod-sel";
    const t=document.createElement("b");t.textContent="Productos";caja.appendChild(t);
    [["p","Petróleo"],["l","LGN y condensado"],["g","Gas natural"]].forEach(([k,txt])=>{
      const lab=document.createElement("label"),ch=document.createElement("input");ch.type="checkbox";ch.checked=sel[k];
      ch.className="leaflet-control-layers-selector";
      ch.addEventListener("change",()=>{sel[k]=ch.checked;aplicar();});
      const s=document.createElement("span");s.textContent=" "+txt;lab.appendChild(ch);lab.appendChild(s);caja.appendChild(lab);});
    lista.appendChild(caja);L.DomEvent.disableClickPropagation(caja);
  }
  aplicar();
  // para los complementos del mapa con plugins (buscador, línea de tiempo)
  window.ppProductos={mapa:mapa,lotes:lotes,sel:sel,aplicar:aplicar,circulos:()=>circulos,COL:COL};
  // al prender una capa del selector, Leaflet la dibuja encima de todo y tapa los lotes: se los vuelve a poner arriba
  const arriba=()=>{lotes.forEach(o=>o.capa.bringToFront());circulos.eachLayer(c=>c.bringToFront());};
  mapa.on("overlayadd baselayerchange",()=>setTimeout(arriba,0));
}
if(document.readyState==="complete")setTimeout(iniciar,0);else window.addEventListener("load",()=>setTimeout(iniciar,0));
})();
</script>
'''
_pj = (PRODUCTOS_HTML.replace("__EST__", json.dumps(EST_PRODUCTO, ensure_ascii=False))
       .replace("__CENTROS__", json.dumps(CENTROS, ensure_ascii=False)).replace("</script>\n</", "</script>\n</"))
m.get_root().html.add_child(folium.Element("{% raw %}" + _pj + "{% endraw %}"))
p = os.path.join(ruta, "PP_mapa_lotes.html"); m.save(p)
subir(p, MAPA_FOLDER_ID, mapa_drive, mime="text/html")
print(f"☁️  Mapa guardado en DATA/MAPA/PP_mapa_lotes.html ({os.path.getsize(p) / 1e6:.1f} MB). "
      "Drive no muestra mapas interactivos: descárgalo y ábrelo con doble clic en Chrome/Edge.")
m

In [ ]:
# ═══ BLOQUE 5 · PLUGINS: FONDOS CARTO, NOMBRES, BUSCADOR, PANTALLA COMPLETA, MEDICIÓN Y LÍNEA DE TIEMPO ═══
# Usa el mapa `m` del Bloque 4 (ejecútalo antes) y lo guarda aparte como DATA/MAPA/PP_mapa_lotes_plus.html.
import calendar
from folium import plugins
# El control de capas tiene que ir al final del HTML (si no, no ve las capas nuevas): se saca y se vuelve a poner
_lc = [k for k, v in list(m._children.items()) if isinstance(v, folium.LayerControl)]
_ctrl = m._children.pop(_lc[0]) if _lc else folium.LayerControl(collapsed=False)
# el Bloque 4 ya guardó el mapa una vez: su script quedó en esa posición del HTML; se quita para que vaya al final
m.get_root().script._children.pop(_ctrl.get_name(), None)

# 1. Fondos CARTO (claro y oscuro): los datos se leen mejor sobre un fondo neutro
_CARTO = "https://{{s}}.basemaps.cartocdn.com/{}/{{z}}/{{x}}/{{y}}{{r}}.png"
_ATTR = "&copy; OpenStreetMap contributors &copy; CARTO"
folium.TileLayer(_CARTO.format("light_all"), attr=_ATTR, name="Claro (CARTO)", subdomains="abcd", show=False).add_to(m)
folium.TileLayer(_CARTO.format("dark_all"), attr=_ATTR, name="Oscuro (CARTO)", subdomains="abcd", show=False).add_to(m)

# 2. Nombres de los lotes (aparecen desde el zoom 7)
etq = folium.FeatureGroup(name="Nombres de lotes", show=True)
_vistos = set()
for _, r in capa_l.iterrows():
    if r["LOTE"] in _vistos or r.geometry is None or r.geometry.is_empty:
        continue
    _vistos.add(r["LOTE"])
    pt = r.geometry.representative_point()
    folium.Marker([pt.y, pt.x], interactive=False, keyboard=False,
                  icon=folium.DivIcon(html=f"<span class='pp-etq'>{_h.escape(str(r['LOTE']))}</span>",
                                      icon_size=(0, 0), icon_anchor=(0, 0))).add_to(etq)
etq.add_to(m)

# 3. Comunidades nativas: grupos pequeños y discretos (morado) en vez de los círculos grandes por defecto
if "mc" in globals() and isinstance(mc, MarkerCluster):
    mc.icon_create_function = """function(c){var n=c.getChildCount(),s=n<10?18:n<100?22:26;
      return L.divIcon({html:'<div style="width:'+s+'px;height:'+s+'px;line-height:'+s+'px;border-radius:50%;'+
      'background:rgba(142,36,170,.75);color:#fff;font:600 10px sans-serif;text-align:center;border:1px solid #fff">'+n+'</div>',
      className:'',iconSize:[s,s]});}"""

# 4–5. Pantalla completa y medición de distancias y áreas
plugins.Fullscreen(position="topleft", title="Pantalla completa", title_cancel="Salir de pantalla completa").add_to(m)
plugins.MeasureControl(position="topleft", primary_length_unit="kilometers", secondary_length_unit="meters",
                       primary_area_unit="hectares", secondary_area_unit="sqmeters").add_to(m)

# 6–7. Buscador de lotes y línea de tiempo animada (producción anual por campo, en bpd / MMPCD)
def serie_anual(prod, gas):
    if prod is None or not len(prod):
        return None
    p_ = prod.copy()
    p_["BARRILES"] = pd.to_numeric(p_["BARRILES"], errors="coerce").fillna(0)
    pr = p_["PRODUCTO"].astype(str).str.upper()
    p_["_K"] = pr.map(lambda x: "p" if x.startswith("PETR") else "l")
    p_["_V"] = p_["BARRILES"]
    partes = [p_]
    if gas is not None and len(gas):
        g_ = gas.copy()
        g_["_K"] = "g"
        g_["_V"] = pd.to_numeric(g_["MPC"], errors="coerce").fillna(0) / 1000      # MMPC
        partes.append(g_)
    t = pd.concat([x[[c for c in ("AÑO", "MES", "LOTE", "LOTE_HISTORICO", "CAMPO", "_K", "_V") if c in x]] for x in partes],
                  ignore_index=True)
    base = t["LOTE_HISTORICO"] if "LOTE_HISTORICO" in t else t["LOTE"]
    t["_C"] = (t["CAMPO"] if "CAMPO" in t else base).fillna(base).astype(str)
    t = t[t["_V"] > 0]
    if t.empty:
        return None
    t["AÑO"] = t["AÑO"].astype(int)
    anios = list(range(int(t["AÑO"].min()), int(t["AÑO"].max()) + 1))
    ult = anios[-1]
    meses_ult = int(t.loc[t["AÑO"] == ult, "MES"].max())
    dias = {a: (366 if calendar.isleap(a) else 365) for a in anios}
    dias[ult] = sum(calendar.monthrange(ult, k)[1] for k in range(1, meses_ult + 1))
    tab = t.groupby(["_C", "_K", "AÑO"])["_V"].sum()
    c = {}
    for (campo, k), s in tab.groupby(level=[0, 1]):
        s = s.droplevel([0, 1])
        c.setdefault(campo, {})[k] = [round(float(s.get(a, 0)) / dias[a], 3 if k == "g" else 0) for a in anios]
    return {"anios": anios, "parcial": meses_ult < 12, "c": c}

TL = serie_anual(prod, globals().get("gas"))
PLUGINS_HTML = r'''<style>
.pp-etq{font:600 11px sans-serif;color:#222;white-space:nowrap;text-shadow:0 0 3px #fff,0 0 3px #fff,0 0 2px #fff;
  pointer-events:none;transform:translate(-50%,-50%);display:inline-block}
.pp-zoom-bajo .pp-etq{display:none}
.pp-buscar{background:#fff;padding:4px 6px;border-radius:5px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:12px sans-serif}
.pp-buscar input{width:150px;border:1px solid #bbb;border-radius:4px;padding:3px 5px;font:12px sans-serif}
.pp-buscar .pp-msg{color:#b71c1c;font-size:11px;margin-top:2px;display:none}
#pp-tl{position:absolute;left:50%;bottom:22px;transform:translateX(-50%);z-index:1000;background:#fff;border-radius:8px;
  box-shadow:0 1px 6px rgba(0,0,0,.35);padding:6px 10px;font:12px sans-serif;color:#222;max-width:calc(100% - 340px)}
#pp-tl button{border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;padding:3px 8px;cursor:pointer;font:600 12px sans-serif}
#pp-tl .pp-fila{display:flex;align-items:center;gap:8px;flex-wrap:wrap}
#pp-tl input[type=range]{width:260px;max-width:40vw}
#pp-tl .pp-anio{font:700 16px sans-serif;min-width:44px;text-align:center}
#pp-tl .pp-esc{display:flex;gap:6px;align-items:center;margin-top:4px;color:#444;font-size:11px;flex-wrap:wrap}
#pp-tl .pp-esc i{display:inline-block;width:12px;height:12px;border-radius:2px;margin-right:3px;vertical-align:middle}
/* escala por encima del botón «Producción fiscalizada» (abajo a la izquierda) */
.leaflet-bottom.leaflet-left{bottom:50px}
@media (max-width:700px){#pp-tl{left:10px;right:10px;transform:none;max-width:none;bottom:60px}#pp-tl input[type=range]{flex:1;min-width:120px;width:auto}}
</style>
<script>
(function(){
const TL=__TL__;
// escala secuencial (un solo tono, de claro a oscuro) del bep/d anual; 0 = gris "sin producción"
const CORTES=[1,1000,5000,20000,50000], TONOS=["#c8e6c9","#81c784","#43a047","#2e7d32","#1b5e20"];
const GRIS="#9e9e9e";
function num(v){return Math.round(v).toString().replace(/\B(?=(\d{3})+(?!\d))/g,".");}
function color(v){if(!(v>=1))return null;let k=0;for(let i=0;i<CORTES.length;i++)if(v>=CORTES[i])k=i;return TONOS[k];}
function esperar(f){if(window.ppProductos)f(window.ppProductos);else setTimeout(()=>esperar(f),200);}
esperar(function(P){
  const mapa=P.mapa;
  // ── nombres de lotes: solo desde zoom 7 para no saturar ──
  const zoomCls=()=>mapa.getContainer().classList.toggle("pp-zoom-bajo",mapa.getZoom()<7);
  mapa.on("zoomend",zoomCls);zoomCls();
  // ── buscador de lotes ──
  const nombres=[...new Set(P.lotes.map(o=>o.lote))].sort((a,b)=>a.localeCompare(b,"es",{numeric:true}));
  const Buscar=L.Control.extend({options:{position:"topleft"},onAdd:function(){
    const d=L.DomUtil.create("div","pp-buscar");
    d.innerHTML="<input list='pp-lista' placeholder='🔎 Buscar lote (p. ej. 95)' aria-label='Buscar lote'>"+
      "<datalist id='pp-lista'>"+nombres.map(n=>"<option value='"+n+"'>").join("")+"</datalist><div class='pp-msg'>No existe ese lote</div>";
    L.DomEvent.disableClickPropagation(d);L.DomEvent.disableScrollPropagation(d);
    const inp=d.querySelector("input"),msg=d.querySelector(".pp-msg");
    const ir=()=>{const q=inp.value.trim().toUpperCase().replace(/^LOTE\s*/,"");if(!q)return;
      const o=P.lotes.find(o=>o.lote.toUpperCase()===q)||P.lotes.find(o=>o.lote.toUpperCase().startsWith(q));
      msg.style.display=o?"none":"block";if(!o)return;
      const b=o.capa.getBounds();mapa.fitBounds(b,{maxZoom:9,padding:[30,30]});
      setTimeout(()=>o.grupo.openPopup(b.getCenter()),350);};
    inp.addEventListener("change",ir);inp.addEventListener("keydown",e=>{if(e.key==="Enter")ir();});
    return d;}});
  new Buscar().addTo(mapa);
  // ── línea de tiempo animada: producción anual (bep/d) de los productos marcados ──
  if(!TL||!TL.anios||!TL.anios.length)return;
  const caja=document.createElement("div");caja.id="pp-tl";mapa.getContainer().appendChild(caja);
  L.DomEvent.disableClickPropagation(caja);L.DomEvent.disableScrollPropagation(caja);
  const st={activo:false,i:TL.anios.length-1,timer:null};
  function valor(campo,i){const d=TL.c[campo];if(!d)return 0;let s=0;
    if(P.sel.p&&d.p)s+=d.p[i]||0;if(P.sel.l&&d.l)s+=d.l[i]||0;if(P.sel.g&&d.g)s+=(d.g[i]||0)*1000/6;return s;}
  function pintar(){
    P.circulos().clearLayers();
    const a=TL.anios[st.i],parcial=TL.parcial&&st.i===TL.anios.length-1;
    caja.querySelector(".pp-anio").textContent=a;caja.querySelector("input").value=st.i;
    caja.querySelector(".pp-nota").textContent=parcial?"(año en curso: promedio de los meses con datos)":"";
    P.lotes.forEach(o=>{const v=valor(o.campo,st.i),c=color(v);
      o.capa.setStyle(c?{color:c,fillColor:c,opacity:1,fillOpacity:.75,weight:1.2}:{color:GRIS,fillColor:GRIS,opacity:.6,fillOpacity:.15,weight:1});
      o.capa.setTooltipContent("Lote "+o.lote+" · "+a+": "+(v>=1?num(v)+" bep/d":"sin producción"));});
  }
  function parar(){if(st.timer){clearInterval(st.timer);st.timer=null;}caja.querySelector(".pp-play").textContent="▶";}
  function jugar(){if(st.timer){parar();return;}if(st.i>=TL.anios.length-1)st.i=0;pintar();
    caja.querySelector(".pp-play").textContent="⏸";
    st.timer=setInterval(()=>{if(st.i>=TL.anios.length-1){parar();return;}st.i++;pintar();},900);}
  function abrir(){st.activo=true;dibujar();pintar();}
  function cerrar(){parar();st.activo=false;dibujar();P.aplicar();}
  function dibujar(){
    if(!st.activo){caja.innerHTML="<button type='button' class='pp-abrir' title='Colorea los lotes por su producción de cada año'>⏱ Evolución histórica de la producción</button>";
      caja.querySelector(".pp-abrir").onclick=abrir;return;}
    caja.innerHTML="<div class='pp-fila'><button type='button' class='pp-play' title='Reproducir / pausar'>▶</button>"+
      "<span class='pp-anio'></span><input type='range' min='0' max='"+(TL.anios.length-1)+"' step='1' aria-label='Año'>"+
      "<button type='button' class='pp-cerrar' title='Volver al estado actual de los lotes'>✕</button></div>"+
      "<div class='pp-esc'><span>bep/d (productos marcados):</span><span><i style='background:"+GRIS+";opacity:.4'></i>0</span>"+
      CORTES.map((c,k)=>"<span><i style='background:"+TONOS[k]+"'></i>"+(k<CORTES.length-1?num(c)+"–"+num(CORTES[k+1]):"≥ "+num(c))+"</span>").join("")+
      "</div><div class='pp-esc pp-nota' style='color:#777'></div>";
    caja.querySelector(".pp-play").onclick=jugar;caja.querySelector(".pp-cerrar").onclick=cerrar;
    caja.querySelector("input").oninput=e=>{parar();st.i=+e.target.value;pintar();};
  }
  window.ppTL={get activo(){return st.activo;},pintar:pintar};
  dibujar();
});
})();
</script>
'''
_tl = json.dumps(TL, ensure_ascii=False, separators=(",", ":")) if TL else "null"
m.get_root().html.add_child(folium.Element("{% raw %}" + PLUGINS_HTML.replace("__TL__", _tl) + "{% endraw %}"))

m.add_child(_ctrl)                                   # control de capas al final
p = os.path.join(ruta, "PP_mapa_lotes_plus.html"); m.save(p)
subir(p, MAPA_FOLDER_ID, mapa_drive, mime="text/html")
print(f"☁️  Mapa con plugins guardado en DATA/MAPA/PP_mapa_lotes_plus.html ({os.path.getsize(p) / 1e6:.1f} MB). "
      "Descárgalo y ábrelo con doble clic en Chrome/Edge.")
m